# 📊 Analisis Disrupsi Supply Chain Indonesia
**Dataset Artikel Berita Supply Chain (2020–2025)**

Notebook ini terdiri dari dua bagian utama:
1. **Word Cloud per Label** — Kata-kata dominan yang mempengaruhi setiap kategori disrupsi
2. **Analisis Kota & Tahun** — Kota-kota Indonesia dan tahun yang paling sering muncul dalam berita disrupsi

## ⚙️ Import Library & Setup

Semua library diimpor sekali di sini. Pastikan semua package sudah ter-install sebelum menjalankan notebook.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Data & Numerik
import pandas as pd
import numpy as np
import re
import ast
import unicodedata
import traceback
import os
import json
import requests
from collections import Counter

# Visualisasi
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from wordcloud import WordCloud

# Geospasial
import geopandas as gpd

# Style global
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print('✅ Semua library berhasil di-import!')

## 📂 Load & Eksplorasi Data

Memuat dataset artikel dari file Excel dan menampilkan ringkasan distribusi label.

In [ ]:
df = pd.read_excel(r'Dataset\Dataset_Artikel.xlsx')

print(f'📌 Jumlah artikel  : {len(df):,}')
print(f'📌 Jumlah kolom    : {df.shape[1]}')
print(f'📌 Rentang tahun   : {df["tahun"].min()} – {df["tahun"].max()}')
print()
print('📊 Distribusi Label (Kategori_LLM):')
print(df['Kategori_LLM'].value_counts().to_string())

## 📊 Visualisasi Distribusi Label

Menampilkan sebaran jumlah artikel per kategori label (Tidak Ada Disrupsi, Disrupsi Non-Halal, Disrupsi Halal).

In [ ]:
# Visualisasi distribusi label
label_counts = df['Kategori_LLM'].value_counts()
colors_label = ['#2196F3', '#F44336', '#FF9800']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(label_counts.index, label_counts.values, color=colors_label, edgecolor='white', height=0.6)
for bar, val in zip(bars, label_counts.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{val:,} artikel', va='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Jumlah Artikel', fontsize=12)
ax.set_title('Distribusi Label Kategori Disrupsi Supply Chain', fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(0, label_counts.max() * 1.2)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('distribusi_label.png', bbox_inches='tight', dpi=150)
plt.show()

---
# 🌩️ BAGIAN 1 — Word Cloud & Analisis Teks per Label

Visualisasi kata-kata yang paling sering muncul dan mempengaruhi setiap kategori label.

## 🛑 Stopwords & Fungsi Tokenisasi

Definisi stopwords Bahasa Indonesia dan fungsi `get_clean_tokens()` untuk membersihkan teks sebelum dianalisis.

In [ ]:
# ─── Stopwords Bahasa Indonesia ───
STOPWORDS_ID = {
    'yang', 'dan', 'di', 'dengan', 'untuk', 'dari', 'ini', 'itu', 'ke', 'pada',
    'dalam', 'adalah', 'sebagai', 'akan', 'juga', 'oleh', 'tidak', 'ada', 'atau',
    'sudah', 'telah', 'saat', 'bisa', 'dapat', 'lebih', 'agar', 'karena', 'jika',
    'serta', 'namun', 'tetapi', 'bahwa', 'hal', 'para', 'pun', 'sehingga', 'maka',
    'selain', 'saja', 'seperti', 'bagi', 'yaitu', 'antara', 'tersebut', 'masih',
    'sangat', 'harus', 'terkait', 'satu', 'dua', 'tiga', 'empat', 'lima',
    'baca', 'juga', 'besar', 'atas', 'lain', 'baru', 'baik', 'kini', 'menjadi',
    'kata', 'hingga', 'tahun', 'per', 'persen', 'nomor', 'ber', 'lanjutan',
    'sama', 'banyak', 'secara', 'semua', 'setelah', 'sebelum', 'ketika', 'mereka',
    'kita', 'kami', 'saya', 'anda', 'dia', 'ia', 'mereka', 'bukan', 'belum',
    'sedang', 'paling', 'pula', 'lagi', 'kali', 'setiap', 'diri', 'maupun',
    'kepada', 'tentang', 'sebuah', 'menurut', 'dengan', 'mulai', 'melalui',
    'berbagai', 'beberapa', 'seluruh', 'suatu', 'tersebut', 'masing', 'inter',
    'nasional', 'kan', 'kon', 'ter', 'men', 'pem', 'pen', 'ber', 'me', 'se',
    'de', 'an', 'nya', 'kan', 'ian', 'asi', 'saat', 'hari', 'bulan', 'lalu',
    'sambil', 'adapun', 'pihak', 'terdapat', 'maka', 'wib', 'kompascom',
    'baca', 'dan', 'itu', 'satu', 'dua', 'pun', 'ion', 'format', 'trans',
    'yang', 'jadi', 'sudah', 'tapi', 'jelas', 'lainnya', 'lain',
    'past', 'pers', 'terus', 'turut', 'mau', 'punya', 'bakal', 'ada',
    'mengatakan', 'menyebut', 'digelar', 'dikutip', 'dijelaskan', 'diungkapkan',
    'disampaikan', 'menyampaikan', 'menjelaskan', 'membahas', 'menyoroti',
    'mendapatkan', 'menggunakan', 'memanfaatkan', 'membantu', 'menghadapi',
    'meningkatkan', 'memberikan', 'memiliki', 'menjalin', 'mengakses',
    'memperkuat', 'mencatat', 'diharapkan', 'memuji', 'melihat', 'menjajaki',
    'dihadiri', 'berlangsung', 'mencakup', 'diolah', 'melacak', 'memfasilitasi',
    'diikuti', 'diadopsi', 'menerapkan', 'diterapkan', 'mengelola', 'berhasil',
    'terhimpun', 'menunjukkan', 'mengubah', 'membentuk', 'penyelesaian',
}

def get_clean_tokens(text):
    """Ekstrak token bersih dari kolom isi_clean."""
    if pd.isna(text):
        return []
    words = text.lower().split()
    return [
        w for w in words
        if w not in STOPWORDS_ID
        and len(w) > 3
        and not w.isdigit()
        and re.match(r'^[a-z]+$', w)
    ]

print('✅ Fungsi preprocessing siap.')

## 🎨 Konfigurasi Warna per Label

Peta warna, background, dan emoji untuk setiap kategori label — digunakan konsisten di seluruh visualisasi.

In [ ]:
# ─── Konfigurasi warna per label ───
LABEL_CONFIG = {
    'Tidak Ada Disrupsi': {
        'colormap': 'Blues',
        'bg_color': '#F0F4FF',
        'title_color': '#1565C0',
        'emoji': '🟢',
    },
    'Disrupsi rantai pasok Umum (Non-Halal)': {
        'colormap': 'Reds',
        'bg_color': '#FFF0F0',
        'title_color': '#B71C1C',
        'emoji': '🔴',
    },
    'Disrupsi Rantai pasok halal': {
        'colormap': 'Oranges',
        'bg_color': '#FFF8E1',
        'title_color': '#E65100',
        'emoji': '🟠',
    },
}

print('✅ Konfigurasi label siap.')

## ☁️ Generate Word Cloud per Label

Membuat satu word cloud terpisah untuk setiap label kategori berdasarkan frekuensi token bersih.

In [ ]:
# ─── Generate Word Cloud per Label (Satu Gambar per Label) ───

for label in [
    'Tidak Ada Disrupsi',
    'Disrupsi rantai pasok Umum (Non-Halal)',
    'Disrupsi Rantai pasok halal'
]:
    cfg = LABEL_CONFIG[label]
    subset = df[df['Kategori_LLM'] == label]['isi_clean']
    n_artikel = len(subset)

    # Gabungkan semua teks
    all_tokens = []
    for text in subset:
        all_tokens.extend(get_clean_tokens(str(text)))

    word_freq = Counter(all_tokens)

    # Generate wordcloud
    wc = WordCloud(
        width=900,
        height=700,
        background_color='white',
        colormap=cfg['colormap'],
        max_words=120,
        min_font_size=9,
        max_font_size=90,
        prefer_horizontal=0.85,
        collocations=False,
        random_state=42,
    ).generate_from_frequencies(word_freq)

    # ── Buat figure terpisah per label ──
    fig, ax = plt.subplots(figsize=(12, 9))
    fig.patch.set_facecolor('#1A1A2E')

    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')

    # Judul label
    short_label = label.replace('Disrupsi rantai pasok Umum (Non-Halal)', 'Disrupsi Umum (Non-Halal)')
    short_label = short_label.replace('Disrupsi Rantai pasok halal', 'Disrupsi Halal')

    ax.set_title(
        f"{cfg['emoji']} {short_label}\n({n_artikel:,} artikel)",
        fontsize=15,
        fontweight='bold',
        color='white',
        pad=14,
    )

    fig.suptitle(
        '🌐 Word Cloud Analisis Disrupsi Supply Chain',
        fontsize=16, fontweight='bold', color='white', y=1.02
    )

    plt.tight_layout(pad=2)

    # ── Nama file aman (tanpa karakter khusus) ──
    safe_name = (
        label
        .replace(' ', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('/', '_')
    )
    filename = f'wordcloud_{safe_name}.png'

    plt.savefig(filename, bbox_inches='tight', dpi=150, facecolor='#1A1A2E')
    plt.show()
    plt.close(fig)  # Tutup figure agar tidak overlap

    print(f"✅ Word cloud tersimpan: {filename}")
    print(f"  {cfg['emoji']} [{label}] — Top 10 kata:")
    for word, count in word_freq.most_common(10):
        print(f"     {word:<25} {count:>5}x")
    print()

## 📊 Bar Chart Top 20 Kata per Label

Menampilkan 20 kata dengan frekuensi tertinggi per label dalam bentuk horizontal bar chart.

In [ ]:
# ─── Bar Chart Top 20 Kata per Label (Satu Gambar per Label) ───

target_labels = [
    'Tidak Ada Disrupsi',
    'Disrupsi rantai pasok Umum (Non-Halal)',
    'Disrupsi Rantai pasok halal',
]

for label in target_labels:
    cfg = LABEL_CONFIG[label]
    subset = df[df['Kategori_LLM'] == label]['isi_clean']

    all_tokens = []
    for text in subset:
        all_tokens.extend(get_clean_tokens(str(text)))

    top20 = Counter(all_tokens).most_common(20)
    words, freqs = zip(*top20)

    # ── Buat figure terpisah per label ──
    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor('#FAFAFA')

    colors = plt.cm.get_cmap(cfg['colormap'])(np.linspace(0.4, 0.85, 20))
    bars = ax.barh(list(reversed(words)), list(reversed(freqs)),
                   color=list(reversed(colors)), edgecolor='white')

    for bar, val in zip(bars, reversed(freqs)):
        ax.text(bar.get_width() + max(freqs) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{val:,}', va='center', fontsize=8.5)

    short = label.replace('Disrupsi rantai pasok Umum (Non-Halal)', 'Disrupsi Umum (Non-Halal)')
    short = short.replace('Disrupsi Rantai pasok halal', 'Disrupsi Halal')

    ax.set_title(f"{cfg['emoji']} {short}", fontsize=13, fontweight='bold',
                 color=cfg['title_color'], pad=12)
    ax.set_xlabel('Frekuensi Kata', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=9)

    fig.suptitle(
        '📊 Top 20 Kata Paling Berpengaruh per Label Disrupsi Supply Chain',
        fontsize=14, fontweight='bold', y=1.02
    )

    plt.tight_layout(pad=2)

    # ── Nama file aman ──
    safe_name = (
        label
        .replace(' ', '_')
        .replace('(', '')
        .replace(')', '')
        .replace('/', '_')
    )
    filename = f'top20_kata_{safe_name}.png'

    plt.savefig(filename, bbox_inches='tight', dpi=150, facecolor='#FAFAFA')
    plt.show()
    plt.close(fig)

    print(f"✅ Bar chart tersimpan: {filename}")

---
# 🗺️ BAGIAN 2 — Analisis Kota & Tahun

Kota mana yang paling sering disebut dalam berita disrupsi supply chain di Indonesia, dan bagaimana tren per tahunnya?

## 🏙️ Daftar Kota & Provinsi Indonesia

Data referensi berisi daftar nama kota dan provinsi se-Indonesia yang digunakan untuk proses ekstraksi lokasi dari teks artikel.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ─── Daftar Kota & Provinsi Indonesia ────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────

PROVINSI_INDONESIA = [
    'aceh', 'sumatera utara', 'sumatera barat', 'riau', 'kepulauan riau',
    'jambi', 'sumatera selatan', 'bangka belitung', 'bengkulu', 'lampung',
    'dki jakarta', 'jakarta', 'banten', 'jawa barat', 'jawa tengah',
    'daerah istimewa yogyakarta', 'yogyakarta', 'jawa timur', 'bali',
    'nusa tenggara barat', 'nusa tenggara timur', 'kalimantan barat',
    'kalimantan tengah', 'kalimantan selatan', 'kalimantan timur',
    'kalimantan utara', 'sulawesi utara', 'sulawesi tengah', 'sulawesi selatan',
    'sulawesi tenggara', 'gorontalo', 'sulawesi barat', 'maluku', 'maluku utara',
    'papua', 'papua barat', 'papua selatan', 'papua tengah', 'papua pegunungan',
    'papua barat daya',
]

KOTA_INDONESIA = [
    # ── Aceh ──────────────────────────────────────────────────
    'banda aceh', 'aceh', 'lhokseumawe', 'langsa', 'sabang', 'subulussalam',
    'meulaboh', 'bireuen', 'sigli', 'takengon', 'blangkejeren', 'tapaktuan',
    'singkil', 'simeulue', 'aceh besar', 'aceh utara', 'aceh timur',
    'aceh selatan', 'aceh tengah', 'aceh barat', 'aceh tenggara',
    'gayo lues', 'nagan raya', 'pidie', 'pidie jaya', 'bener meriah',
    'aceh barat daya', 'aceh jaya',

    # ── Sumatera Utara ────────────────────────────────────────
    'medan', 'binjai', 'sibolga', 'tanjungbalai', 'pematangsiantar',
    'tebing tinggi', 'padangsidimpuan', 'gunungsitoli', 'deli serdang',
    'langkat', 'karo', 'simalungun', 'dairi', 'tapanuli utara',
    'tapanuli selatan', 'tapanuli tengah', 'nias', 'nias selatan',
    'nias utara', 'nias barat', 'mandailing natal', 'labuhanbatu',
    'asahan', 'batu bara', 'serdang bedagai', 'padang lawas',
    'padang lawas utara', 'humbang hasundutan', 'toba samosir', 'samosir',
    'pakpak bharat', 'labuhanbatu utara', 'labuhanbatu selatan',

    # ── Sumatera Barat ────────────────────────────────────────
    'padang', 'bukittinggi', 'payakumbuh', 'solok', 'sawahlunto',
    'padangpanjang', 'pariaman', 'agam', 'tanah datar', 'limapuluh kota',
    'pasaman', 'pasaman barat', 'sijunjung', 'dharmasraya', 'solok selatan',
    'pesisir selatan', 'kepulauan mentawai',

    # ── Riau ──────────────────────────────────────────────────
    'pekanbaru', 'dumai', 'bengkalis', 'siak', 'kampar', 'rokan hulu',
    'rokan hilir', 'indragiri hulu', 'indragiri hilir', 'pelalawan',
    'kuantan singingi', 'kepulauan meranti',

    # ── Kepulauan Riau ────────────────────────────────────────
    'tanjungpinang', 'batam', 'bintan', 'karimun', 'natuna', 'anambas', 'lingga',

    # ── Jambi ─────────────────────────────────────────────────
    'jambi', 'sungai penuh', 'muaro jambi', 'batanghari', 'bungo', 'tebo',
    'merangin', 'sarolangun', 'kerinci', 'tanjung jabung barat',
    'tanjung jabung timur',

    # ── Sumatera Selatan ──────────────────────────────────────
    'palembang', 'prabumulih', 'pagaralam', 'lubuklinggau', 'lahat',
    'muara enim', 'ogan komering ilir', 'ogan komering ulu', 'musi banyuasin',
    'musi rawas', 'musi rawas utara', 'empat lawang', 'banyuasin',
    'penukal abab', 'ogan ilir', 'ogan komering ulu timur',
    'ogan komering ulu selatan',

    # ── Bangka Belitung ───────────────────────────────────────
    'pangkalpinang', 'bangka', 'belitung', 'bangka barat', 'bangka tengah',
    'bangka selatan', 'belitung timur',

    # ── Bengkulu ──────────────────────────────────────────────
    'bengkulu', 'rejang lebong', 'kepahiang', 'lebong', 'bengkulu utara',
    'bengkulu tengah', 'bengkulu selatan', 'seluma', 'kaur', 'muko muko',

    # ── Lampung ───────────────────────────────────────────────
    'bandar lampung', 'metro', 'lampung utara', 'lampung selatan',
    'lampung tengah', 'lampung barat', 'lampung timur', 'mesuji',
    'tulang bawang', 'tulang bawang barat', 'pesawaran', 'pringsewu',
    'tanggamus', 'way kanan', 'pesisir barat',

    # ── Banten ────────────────────────────────────────────────
    'serang', 'cilegon', 'tangerang', 'tangerang selatan', 'lebak', 'pandeglang',

    # ── DKI Jakarta ───────────────────────────────────────────
    'jakarta', 'jakarta pusat', 'jakarta utara', 'jakarta barat',
    'jakarta selatan', 'jakarta timur', 'kepulauan seribu',

    # ── Jawa Barat ────────────────────────────────────────────
    'bandung', 'bekasi', 'bogor', 'depok', 'cimahi', 'sukabumi', 'cirebon',
    'tasikmalaya', 'banjar', 'karawang', 'cikarang', 'cikampek', 'purwakarta',
    'subang', 'garut', 'cianjur', 'kuningan', 'majalengka', 'indramayu',
    'sumedang', 'pangandaran', 'bandung barat',

    # ── Jawa Tengah ───────────────────────────────────────────
    'semarang', 'surakarta', 'solo', 'salatiga', 'magelang', 'pekalongan',
    'tegal', 'purwokerto', 'cilacap', 'banyumas', 'kebumen', 'purworejo',
    'wonosobo', 'temanggung', 'kendal', 'demak', 'jepara', 'kudus', 'pati',
    'rembang', 'blora', 'grobogan', 'sragen', 'karanganyar', 'wonogiri',
    'sukoharjo', 'boyolali', 'klaten', 'brebes', 'pemalang', 'batang',
    'banjarnegara', 'purbalingga', 'gombong',

    # ── DI Yogyakarta ─────────────────────────────────────────
    'yogyakarta', 'sleman', 'bantul', 'kulonprogo', 'gunung kidul', 'wates',

    # ── Jawa Timur ────────────────────────────────────────────
    'surabaya', 'malang', 'kediri', 'blitar', 'madiun', 'mojokerto',
    'pasuruan', 'probolinggo', 'batu', 'jember', 'banyuwangi', 'situbondo',
    'bondowoso', 'lumajang', 'jombang', 'nganjuk', 'tulungagung', 'trenggalek',
    'pacitan', 'ponorogo', 'magetan', 'ngawi', 'bojonegoro', 'lamongan',
    'gresik', 'sidoarjo', 'tuban', 'bangkalan', 'sampang', 'pamekasan', 'sumenep',

    # ── Bali ──────────────────────────────────────────────────
    'denpasar', 'badung', 'gianyar', 'tabanan', 'klungkung', 'karangasem',
    'buleleng', 'jembrana', 'bangli', 'kuta', 'ubud', 'sanur', 'nusa dua',
    'singaraja',
    # 'negara' → DIHAPUS PERMANEN (artinya "country/state")

    # ── Nusa Tenggara Barat ───────────────────────────────────
    'mataram', 'bima', 'dompu', 'sumbawa', 'sumbawa besar', 'lombok',
    'lombok barat', 'lombok tengah', 'lombok timur', 'lombok utara', 'raba',

    # ── Nusa Tenggara Timur ───────────────────────────────────
    'kupang', 'ende', 'maumere', 'bajawa', 'ruteng', 'labuan bajo',
    'waingapu', 'waikabubak', 'atambua', 'kefamenanu', 'soe',
    'manggarai', 'manggarai barat', 'manggarai timur', 'nagekeo', 'ngada',
    'sikka', 'alor', 'lembata', 'flores timur', 'rote ndao',
    'sumba barat', 'sumba timur', 'sumba tengah', 'sumba barat daya',
    'timor tengah utara', 'timor tengah selatan', 'belu', 'malaka',
    'sabu raijua',

    # ── Kalimantan Barat ──────────────────────────────────────
    'pontianak', 'singkawang', 'sambas', 'mempawah', 'bengkayang', 'landak',
    'sanggau', 'sekadau', 'sintang', 'melawi', 'kapuas hulu', 'ketapang',
    'kayong utara', 'kubu raya',

    # ── Kalimantan Tengah ─────────────────────────────────────
    'palangka raya', 'sampit', 'pangkalan bun', 'muara teweh',
    'kuala kapuas', 'buntok', 'tamiang layang', 'kotawaringin barat',
    'kotawaringin timur', 'barito utara', 'barito selatan', 'barito timur',
    'pulang pisau', 'gunung mas', 'katingan', 'lamandau', 'seruyan', 'sukamara',

    # ── Kalimantan Selatan ────────────────────────────────────
    'banjarmasin', 'banjarbaru', 'martapura', 'amuntai', 'kandangan',
    'pelaihari', 'batulicin', 'kotabaru', 'tanjung', 'rantau',
    'barito kuala', 'hulu sungai utara', 'hulu sungai tengah',
    'hulu sungai selatan', 'tabalong', 'tapin', 'tanah laut',
    'tanah bumbu', 'balangan', 'banjar',

    # ── Kalimantan Timur ──────────────────────────────────────
    'samarinda', 'balikpapan', 'bontang', 'tarakan', 'tenggarong', 'sangatta',
    'kutai kartanegara', 'kutai barat', 'kutai timur', 'berau',
    'penajam paser utara', 'paser', 'mahakam ulu',

    # ── Kalimantan Utara ──────────────────────────────────────
    'tanjung selor', 'nunukan', 'malinau', 'bulungan', 'tana tidung',

    # ── Sulawesi Utara ────────────────────────────────────────
    'manado', 'bitung', 'tomohon', 'kotamobagu', 'minahasa',
    'minahasa utara', 'minahasa selatan', 'minahasa tenggara',
    'bolaang mongondow', 'bolaang mongondow utara', 'bolaang mongondow timur',
    'bolaang mongondow selatan', 'sangihe', 'sitaro', 'talaud',

    # ── Sulawesi Tengah ───────────────────────────────────────
    'palu', 'luwuk', 'toli toli', 'buol', 'donggala', 'sigi',
    'parigi moutong', 'poso', 'tojo una una', 'banggai',
    'banggai kepulauan', 'morowali', 'morowali utara', 'banggai laut',

    # ── Sulawesi Selatan ──────────────────────────────────────
    'makassar', 'parepare', 'palopo', 'maros', 'gowa', 'takalar',
    'jeneponto', 'bantaeng', 'bulukumba', 'sinjai', 'bone', 'soppeng',
    'wajo', 'sidrap', 'pinrang', 'enrekang', 'tana toraja', 'toraja utara',
    'luwu', 'luwu utara', 'luwu timur', 'selayar', 'barru', 'pangkep',
    'kepulauan selayar',

    # ── Sulawesi Tenggara ─────────────────────────────────────
    'kendari', 'baubau', 'kolaka', 'kolaka utara', 'kolaka timur',
    'konawe', 'konawe selatan', 'konawe utara', 'konawe kepulauan',
    'muna', 'muna barat', 'buton', 'buton utara', 'buton selatan',
    'buton tengah', 'bombana', 'wakatobi', 'raha',

    # ── Gorontalo ─────────────────────────────────────────────
    'gorontalo', 'gorontalo utara', 'bone bolango', 'boalemo',
    'pohuwato', 'gorontalo kota',

    # ── Sulawesi Barat ────────────────────────────────────────
    'mamuju', 'majene', 'polewali mandar', 'mamasa', 'mamuju utara',
    'mamuju tengah', 'pasangkayu',

    # ── Maluku ────────────────────────────────────────────────
    'ambon', 'tual', 'masohi', 'saumlaki', 'namlea', 'bula', 'dobo',
    'maluku tengah', 'maluku tenggara', 'maluku barat daya',
    'seram bagian barat', 'seram bagian timur', 'kepulauan aru',
    'buru', 'buru selatan',

    # ── Maluku Utara ──────────────────────────────────────────
    'ternate', 'tidore', 'tobelo', 'labuha', 'sanana', 'sofifi',
    'halmahera utara', 'halmahera selatan', 'halmahera barat',
    'halmahera timur', 'halmahera tengah', 'kepulauan sula',
    'pulau morotai', 'pulau taliabu',

    # ── Papua ─────────────────────────────────────────────────
    'jayapura', 'sentani', 'abepura', 'sarmi', 'keerom', 'waropen',
    'supiori', 'mamberamo raya', 'yalimo', 'jayawijaya',
    'pegunungan bintang', 'tolikara', 'nduga', 'lanny jaya',
    'mamberamo tengah',

    # ── Papua Barat ───────────────────────────────────────────
    'manokwari', 'fakfak', 'kaimana', 'teluk bintuni', 'teluk wondama',
    'manokwari selatan', 'pegunungan arfak', 'tambrauw', 'maybrat',
    'raja ampat',

    # ── Papua Selatan ─────────────────────────────────────────
    'merauke', 'boven digoel', 'mappi', 'asmat',

    # ── Papua Tengah ──────────────────────────────────────────
    'nabire', 'paniai', 'dogiyai', 'intan jaya', 'deiyai', 'timika',
    'puncak jaya', 'puncak',

    # ── Papua Pegunungan ──────────────────────────────────────
    'wamena',

    # ── Papua Barat Daya ──────────────────────────────────────
    'sorong', 'sorong selatan',
]

WILAYAH_INDONESIA = sorted(set(PROVINSI_INDONESIA + KOTA_INDONESIA))

print(f'✅ Total {len(PROVINSI_INDONESIA)} provinsi dimuat.')
print(f'✅ Total {len(set(KOTA_INDONESIA))} kota/kabupaten dimuat.')
print(f'✅ Total gabungan: {len(WILAYAH_INDONESIA)} wilayah.')


# ─────────────────────────────────────────────────────────────────────────────
# ─── Logika Deteksi Kota ─────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────

# ── Kata ambigu: hanya dihitung jika ada kata konteks LOKASI di sekitarnya ───
AMBIGUOUS_CITIES = {
    'batu',         # = stone/rock     (vs Kota Batu, Jatim)
    'batu bara',    # = coal           (vs Kab. Batu Bara, Sumut)
    'tanjung',      # = cape/headland  (vs banyak kota bernama Tanjung...)
    'puncak',       # = peak/summit    (vs Kab. Puncak, Papua)
    'solo',         # = alone          (vs Kota Solo / Surakarta)
    'banjar',       # = row/line       (vs Kota Banjar / Kab. Banjar)
    'wates',        # = border/limit   (vs Kota Wates, Kulon Progo)
    'raba',         # pendek & ambigu
    'ende',         # pendek & ambigu
    'soe',          # pendek & ambigu
    'raha',         # pendek & ambigu
    'dobo',         # pendek & ambigu
    'bula',         # = bubble/blister
    'bone',         # pendek, bisa false positive
    'muara',        # = river mouth
}

# ── Kata konteks LOKASI ───────────────────────────────────────────────────────
LOCATION_MARKERS = {
    'di', 'dari', 'ke', 'menuju', 'via', 'melalui', 'dekat',
    'kota', 'kabupaten', 'kab', 'provinsi', 'prov', 'wilayah', 'daerah',
    'kawasan', 'area', 'distrik', 'kecamatan', 'lokasi', 'site',
    'kantor', 'pabrik', 'cabang', 'pusat', 'gudang', 'pelabuhan',
    'bandara', 'terminal', 'stasiun', 'fasilitas', 'plant', 'hub', 'depot',
}

# ── Kata konteks KEJADIAN: sesuatu terjadi/berdampak di lokasi tersebut ──────
EVENT_MARKERS = {
    # kejadian fisik / bencana
    'terjadi', 'berlangsung', 'melanda', 'menimpa', 'dilanda', 'diguncang',
    'banjir', 'longsor', 'gempa', 'kebakaran', 'bencana', 'musibah',
    'kecelakaan', 'ledakan', 'kebocoran', 'tumpahan', 'angin', 'topan',
    # dampak supply chain / logistik
    'terdampak', 'mengalami', 'terganggu', 'terhenti', 'tertunda', 'terlambat',
    'terputus', 'lumpuh', 'macet', 'ditutup', 'diblokir', 'mogok', 'blokir',
    'hambat', 'gangguan', 'disrupsi', 'kekurangan', 'kelangkaan', 'lonjakan',
    'penurunan', 'anjlok', 'kenaikan', 'tersendat', 'terhambat',
    # operasional
    'beroperasi', 'berproduksi', 'berpusat', 'berlokasi', 'terletak',
    'berada', 'dibangun', 'diresmikan', 'dibuka', 'diinvestasikan', 'dikelola',
    # aksi / kebijakan
    'ditetapkan', 'diberlakukan', 'diumumkan', 'diluncurkan', 'diterapkan',
    'digelar', 'dilakukan', 'dilaksanakan', 'diadakan',
}

CONTEXT_WINDOW = 6

_all_cities_set        = set(KOTA_INDONESIA) | set(PROVINSI_INDONESIA)
HIGH_CONFIDENCE_CITIES = _all_cities_set - AMBIGUOUS_CITIES
_sorted_candidates     = sorted(
    HIGH_CONFIDENCE_CITIES | AMBIGUOUS_CITIES, key=len, reverse=True
)


def _normalize(text):
    return re.sub(r'\s+', ' ', str(text).lower().strip())

def _context_tokens(text, start, end, window=CONTEXT_WINDOW):
    left  = text[:start].split()[-window:]
    right = text[end:].split()[:window]
    return {t.strip('.,;:()"\'') for t in left + right}

def _has_location_context(text, start, end):
    return bool(_context_tokens(text, start, end) & LOCATION_MARKERS)

def _has_event_context(text, start, end):
    return bool(_context_tokens(text, start, end) & EVENT_MARKERS)

def extract_kota_kejadian(text):
    """
    Ekstrak kota yang muncul dalam konteks kejadian/disrupsi/operasional.
    Kata ambigu tetap butuh konteks lokasi dulu, lalu dicek konteks kejadian.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    text_norm         = _normalize(text)
    found             = set()
    matched_positions = set()

    for city in _sorted_candidates:
        pattern = r'(?<!\w)' + re.escape(city) + r'(?!\w)'
        for m in re.finditer(pattern, text_norm):
            positions = set(range(m.start(), m.end()))
            if positions & matched_positions:
                continue
            # Kata ambigu: harus ada konteks lokasi
            if city in AMBIGUOUS_CITIES:
                if not _has_location_context(text_norm, m.start(), m.end()):
                    continue
            # Semua kota: harus ada konteks kejadian
            if not _has_event_context(text_norm, m.start(), m.end()):
                continue
            found.add(city)
            matched_positions |= positions

    return sorted(found)


# ─── Terapkan ke DataFrame ────────────────────────────────────────────────────
TEXT_COL = 'isi_bersih_v3'

# df_non_disrupsi = semua artikel yang BUKAN disrupsi
df_non_disrupsi = df[df['Kategori_LLM'] == 'Tidak Ada Disrupsi'].copy()
df_disrupsi = df[df['Kategori_LLM'] != 'Tidak Ada Disrupsi'].copy()

df['kota_kejadian']         = df[TEXT_COL].apply(extract_kota_kejadian)
df_disrupsi['kota_kejadian']    = df_disrupsi[TEXT_COL].apply(extract_kota_kejadian)
df_non_disrupsi['kota_kejadian']= df_non_disrupsi[TEXT_COL].apply(extract_kota_kejadian)

print('✅ Kolom kota_kejadian berhasil dibuat untuk ketiga DataFrame.')
print(f'   df         : {len(df)} artikel')
print(f'   df_disrupsi    : {len(df_disrupsi)} artikel')
print(f'   df_non_disrupsi: {len(df_non_disrupsi)} artikel')


# ─────────────────────────────────────────────────────────────────────────────
# ─── Hitung Frekuensi Kota Kejadian ──────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────

counter_all         = Counter(k for lst in df['kota_kejadian']         for k in lst)
counter_disrupsi    = Counter(k for lst in df_disrupsi['kota_kejadian']    for k in lst)
counter_non_disrupsi= Counter(k for lst in df_non_disrupsi['kota_kejadian']for k in lst)

df_kota_all         = pd.DataFrame(counter_all.most_common(30),
                                    columns=['kota', 'frekuensi'])
df_kota_disrupsi    = pd.DataFrame(counter_disrupsi.most_common(30),
                                    columns=['kota', 'frekuensi'])
df_kota_non_disrupsi= pd.DataFrame(counter_non_disrupsi.most_common(30),
                                    columns=['kota', 'frekuensi'])

print('\n🏙️ Top 20 Lokasi Kejadian — Semua Artikel:')
print(df_kota_all.head(20).to_string(index=False))
print('\n🔴 Top 20 Lokasi Kejadian — Artikel Disrupsi:')
print(df_kota_disrupsi.head(20).to_string(index=False))
print('\n🟢 Top 20 Lokasi Kejadian — Artikel Non-Disrupsi:')
print(df_kota_non_disrupsi.head(20).to_string(index=False))


# ─────────────────────────────────────────────────────────────────────────────
# ─── Visualisasi: 3 Chart Terpisah, masing-masing disimpan 1 PNG ─────────────
# ─────────────────────────────────────────────────────────────────────────────

top_n = 25

CHARTS = [
    {
        'df'      : df_kota_all,
        'title'   : '🏙️ Top 25 Lokasi Kejadian Supply Chain\n(Semua Artikel)',
        'color'   : '#42A5F5',
        'filename': 'kota_kejadian_semua.png',
        'label'   : 'Semua Artikel',
    },
    {
        'df'      : df_kota_disrupsi,
        'title'   : '🔴 Top 25 Lokasi Kejadian Supply Chain\n(Artikel Disrupsi)',
        'color'   : '#EF5350',
        'filename': 'kota_kejadian_disrupsi.png',
        'label'   : 'Artikel Disrupsi',
    },
    {
        'df'      : df_kota_non_disrupsi,
        'title'   : '🟢 Top 25 Lokasi Kejadian Supply Chain\n(Artikel Non-Disrupsi)',
        'color'   : '#66BB6A',
        'filename': 'kota_kejadian_non_disrupsi.png',
        'label'   : 'Artikel Non-Disrupsi',
    },
]

for chart in CHARTS:
    data = chart['df'].head(top_n).copy()

    if data.empty:
        print(f'⚠️  Tidak ada data untuk: {chart["filename"]} — dilewati.')
        continue

    fig, ax = plt.subplots(figsize=(13, 9))
    fig.patch.set_facecolor('#FAFAFA')

    y = np.arange(len(data))

    bars = ax.barh(y, data['frekuensi'], height=0.6,
                   color=chart['color'], alpha=0.88, label=chart['label'])

    # Label angka di ujung bar
    for bar in bars:
        w = bar.get_width()
        ax.text(w + 0.3, bar.get_y() + bar.get_height() / 2,
                str(int(w)), va='center', ha='left', fontsize=9, color='#444')

    ax.set_yticks(y)
    ax.set_yticklabels([k.title() for k in data['kota']], fontsize=10)
    ax.set_xlabel('Frekuensi Kemunculan sebagai Lokasi Kejadian', fontsize=11)
    ax.set_title(chart['title'], fontsize=14, fontweight='bold', pad=15)
    ax.legend(fontsize=10, loc='lower right')
    ax.spines[['top', 'right']].set_visible(False)
    ax.invert_yaxis()

    plt.tight_layout()
    plt.savefig(chart['filename'], bbox_inches='tight', dpi=150)
    plt.show()
    print(f'✅ Tersimpan: {chart["filename"]}')

## 🔍 Ekstraksi Kota dari Teks Artikel

Fungsi `extract_cities()` mendeteksi kota yang muncul dalam setiap artikel menggunakan regex word boundary agar tidak ada false match.

In [ ]:
# ─── Ekstraksi Kota dari Teks ───
def extract_cities(text, kota_list):
    """Cari kota-kota yang muncul dalam teks."""
    if pd.isna(text):
        return []
    text_lower = text.lower()
    found = []
    for kota in kota_list:
        # Match kata utuh (word boundary)
        pattern = r'\b' + re.escape(kota) + r'\b'
        if re.search(pattern, text_lower):
            found.append(kota)
    return found

print('⏳ Mendeteksi kota dari seluruh artikel... (bisa 1–2 menit)')

# Hanya dari artikel disrupsi (Non-Halal + Halal)
df_disrupsi = df[df['Kategori_LLM'] != 'Tidak Ada Disrupsi'].copy()
df_all = df.copy()

df_all['kota_disebut'] = df_all['isi_clean'].apply(lambda x: extract_cities(x, KOTA_INDONESIA))
df_disrupsi['kota_disebut'] = df_disrupsi['isi_clean'].apply(lambda x: extract_cities(x, KOTA_INDONESIA))

print(f'✅ Selesai! {len(df_disrupsi)} artikel disrupsi diproses.')

## 📈 Hitung Frekuensi Kota

Menghitung berapa kali setiap kota disebut — baik di seluruh artikel maupun khusus artikel bertanda disrupsi.

In [ ]:
# ─── Hitung Frekuensi Kota ───

# Semua kota dari semua artikel
all_kota_all = [k for kota_list in df_all['kota_disebut'] for k in kota_list]
# Kota dari artikel disrupsi saja
all_kota_disrupsi = [k for kota_list in df_disrupsi['kota_disebut'] for k in kota_list]

counter_all = Counter(all_kota_all)
counter_disrupsi = Counter(all_kota_disrupsi)

df_kota_all = pd.DataFrame(counter_all.most_common(30), columns=['kota', 'frekuensi_semua_artikel'])
df_kota_disrupsi = pd.DataFrame(counter_disrupsi.most_common(30), columns=['kota', 'frekuensi_artikel_disrupsi'])

print('🏙️ Top 20 Kota — Semua Artikel:')
print(df_kota_all.head(20).to_string(index=False))
print()
print('🔴 Top 20 Kota — Artikel Disrupsi Saja:')
print(df_kota_disrupsi.head(20).to_string(index=False))

## 📊 Visualisasi Top 25 Kota (Semua Artikel vs Disrupsi)

Perbandingan frekuensi penyebutan kota antara semua artikel dengan artikel yang tergolong disrupsi supply chain.

In [ ]:
# ─── Visualisasi: Top 25 Kota (Semua Artikel vs Disrupsi) ───
top_n = 25
top_kota = df_kota_all.head(top_n).copy()

# Merge dengan data disrupsi
top_kota = top_kota.merge(df_kota_disrupsi, on='kota', how='left').fillna(0)
top_kota['frekuensi_artikel_disrupsi'] = top_kota['frekuensi_artikel_disrupsi'].astype(int)

fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor('#FAFAFA')

y = np.arange(top_n)
bar_h = 0.38

b1 = ax.barh(y + bar_h/2, top_kota['frekuensi_semua_artikel'], height=bar_h,
             label='Semua Artikel', color='#42A5F5', alpha=0.88)
b2 = ax.barh(y - bar_h/2, top_kota['frekuensi_artikel_disrupsi'], height=bar_h,
             label='Artikel Disrupsi', color='#EF5350', alpha=0.88)

ax.set_yticks(y)
ax.set_yticklabels([k.title() for k in top_kota['kota']], fontsize=10)
ax.set_xlabel('Frekuensi Kemunculan', fontsize=12)
ax.set_title(
    '🏙️ Top 25 Kota Indonesia Paling Sering Disebut dalam Berita Supply Chain\n'
    '(Semua Artikel vs Artikel Disrupsi)',
    fontsize=14, fontweight='bold', pad=15
)
ax.legend(fontsize=11, loc='lower right')
ax.spines[['top', 'right']].set_visible(False)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('kota_frekuensi.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Chart kota tersimpan: kota_frekuensi.png')

## 🏷️ Analisis Kota per Label Disrupsi

Menampilkan top 15 kota yang paling sering muncul, dipisahkan per label disrupsi (Non-Halal dan Halal).

In [ ]:
# ─── Analisis Kota per Label Disrupsi ───
fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.patch.set_facecolor('#FAFAFA')

disrupsi_labels = [
    ('Disrupsi rantai pasok Umum (Non-Halal)', '#EF5350', '🔴'),
    ('Disrupsi Rantai pasok halal', '#FF9800', '🟠'),
]

for ax, (label, color, emoji) in zip(axes, disrupsi_labels):
    subset = df[df['Kategori_LLM'] == label].copy()
    subset['kota_disebut'] = subset['isi_clean'].apply(lambda x: extract_cities(x, KOTA_INDONESIA))
    kota_counts = Counter([k for klist in subset['kota_disebut'] for k in klist])

    if not kota_counts:
        ax.text(0.5, 0.5, 'Tidak ada data kota', ha='center', va='center', transform=ax.transAxes)
        continue

    top15 = kota_counts.most_common(15)
    kota_names, counts = zip(*top15)

    cmap = plt.cm.Reds if 'Non-Halal' in label else plt.cm.Oranges
    bar_colors = cmap(np.linspace(0.4, 0.85, len(kota_names)))

    bars = ax.barh(
        [k.title() for k in reversed(kota_names)],
        list(reversed(counts)),
        color=list(reversed(bar_colors)),
        edgecolor='white'
    )

    for bar, val in zip(bars, reversed(counts)):
        ax.text(bar.get_width() + max(counts)*0.01, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=9)

    short_label = label.replace('Disrupsi rantai pasok Umum (Non-Halal)', 'Disrupsi Umum (Non-Halal)')
    short_label = short_label.replace('Disrupsi Rantai pasok halal', 'Disrupsi Halal')
    ax.set_title(f'{emoji} Top 15 Kota — {short_label}\n({len(subset)} artikel)',
                 fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel('Frekuensi Kemunculan', fontsize=10)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('🗺️ Sebaran Kota dalam Artikel Disrupsi Supply Chain per Kategori',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(pad=2)
plt.savefig('kota_per_label.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Chart kota per label tersimpan: kota_per_label.png')

## 📅 Analisis Distribusi Artikel per Tahun

Menghitung dan menampilkan distribusi jumlah artikel disrupsi supply chain berdasarkan tahun terbit (2020–2025).

In [ ]:
# ─── Analisis Tahun ───

# Frekuensi artikel per tahun — semua vs disrupsi
tahun_all = df['tahun'].value_counts().sort_index()
tahun_disrupsi = df[df['Kategori_LLM'] != 'Tidak Ada Disrupsi']['tahun'].value_counts().sort_index()
tahun_non_halal = df[df['Kategori_LLM'] == 'Disrupsi rantai pasok Umum (Non-Halal)']['tahun'].value_counts().sort_index()
tahun_halal = df[df['Kategori_LLM'] == 'Disrupsi Rantai pasok halal']['tahun'].value_counts().sort_index()

print('📅 Distribusi Artikel per Tahun:')
tahun_df = pd.DataFrame({
    'Semua Artikel': tahun_all,
    'Disrupsi (Total)': tahun_disrupsi,
    'Disrupsi Non-Halal': tahun_non_halal,
    'Disrupsi Halal': tahun_halal,
}).fillna(0).astype(int)
print(tahun_df.to_string())

# Hitung proporsi disrupsi per tahun
tahun_df['Proporsi Disrupsi (%)'] = (tahun_df['Disrupsi (Total)'] / tahun_df['Semua Artikel'] * 100).round(1)
print()
print('📊 Proporsi Disrupsi per Tahun:')
print(tahun_df[['Semua Artikel', 'Disrupsi (Total)', 'Proporsi Disrupsi (%)']].to_string())

## 📊 Visualisasi Panel Tren Tahunan

Empat panel visualisasi: stacked bar, line tren, proporsi per tahun, dan heatmap label vs tahun.

In [ ]:
years = sorted(df['tahun'].unique())

# ── Panel 1: Stacked Bar ──────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(8, 6))
fig1.patch.set_facecolor('#FAFAFA')
tahun_by_label = df.groupby(['tahun', 'Kategori_LLM']).size().unstack(fill_value=0)
tahun_by_label.plot(kind='bar', ax=ax1, stacked=True,
                    color=['#2196F3', '#EF5350', '#FF9800'],
                    edgecolor='white', width=0.7)
ax1.set_title('📊 Jumlah Artikel per Tahun (Stacked per Label)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Tahun', fontsize=11)
ax1.set_ylabel('Jumlah Artikel', fontsize=11)
ax1.tick_params(axis='x', rotation=0)
ax1.legend(title='Label', fontsize=8, title_fontsize=9)
ax1.spines[['top', 'right']].set_visible(False)
for container in ax1.containers:
    ax1.bar_label(container, label_type='center', fontsize=8, color='white', fontweight='bold',
                  fmt=lambda x: str(int(x)) if x > 0 else '')
plt.tight_layout()
plt.savefig('panel1_stacked_bar.png', bbox_inches='tight', dpi=150)
plt.close()
print('✅ Tersimpan: panel1_stacked_bar.png')


# ── Panel 2: Line Chart Tren ─────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(8, 6))
fig2.patch.set_facecolor('#FAFAFA')
colors_line = {'Disrupsi rantai pasok Umum (Non-Halal)': '#EF5350',
               'Disrupsi Rantai pasok halal': '#FF9800',
               'Tidak Ada Disrupsi': '#42A5F5'}
for label, color in colors_line.items():
    data = df[df['Kategori_LLM'] == label]['tahun'].value_counts().sort_index()
    data = data.reindex(years, fill_value=0)
    short = label.replace('Disrupsi rantai pasok Umum (Non-Halal)', 'Disrupsi Umum')
    short = short.replace('Disrupsi Rantai pasok halal', 'Disrupsi Halal')
    ax2.plot(years, data.values, marker='o', label=short, color=color, linewidth=2.5, markersize=7)
    for y_val, x_val in zip(years, data.values):
        ax2.annotate(str(x_val), (y_val, x_val), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=8.5, color=color)
ax2.set_title('📈 Tren Artikel per Label per Tahun', fontsize=12, fontweight='bold')
ax2.set_xlabel('Tahun', fontsize=11)
ax2.set_ylabel('Jumlah Artikel', fontsize=11)
ax2.legend(fontsize=9)
ax2.spines[['top', 'right']].set_visible(False)
ax2.set_xticks(years)
plt.tight_layout()
plt.savefig('panel2_line_tren.png', bbox_inches='tight', dpi=150)
plt.close()
print('✅ Tersimpan: panel2_line_tren.png')


# ── Panel 3: Proporsi Bar ────────────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(8, 6))
fig3.patch.set_facecolor('#FAFAFA')
disrupsi_per_tahun = df[df['Kategori_LLM'] != 'Tidak Ada Disrupsi']['tahun'].value_counts().sort_index()
total_per_tahun = df['tahun'].value_counts().sort_index()
proporsi = (disrupsi_per_tahun / total_per_tahun * 100).reindex(years, fill_value=0).round(1)
bars = ax3.bar(years, proporsi.values, color='#EF5350', alpha=0.8, edgecolor='white', width=0.6)
for bar, pct in zip(bars, proporsi.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{pct}%', ha='center', fontsize=11, fontweight='bold', color='#B71C1C')
ax3.set_title('🔴 Proporsi Artikel Disrupsi per Tahun (%)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Tahun', fontsize=11)
ax3.set_ylabel('Proporsi (%)', fontsize=11)
ax3.set_ylim(0, max(proporsi.values) * 1.3)
ax3.spines[['top', 'right']].set_visible(False)
ax3.set_xticks(years)
plt.tight_layout()
plt.savefig('panel3_proporsi.png', bbox_inches='tight', dpi=150)
plt.close()
print('✅ Tersimpan: panel3_proporsi.png')


# ── Panel 4: Heatmap ─────────────────────────────────────────────
fig4, ax4 = plt.subplots(figsize=(8, 6))
fig4.patch.set_facecolor('#FAFAFA')
pivot = df.groupby(['Kategori_LLM', 'tahun']).size().unstack(fill_value=0)
pivot.index = [
    idx.replace('Disrupsi rantai pasok Umum (Non-Halal)', 'Disrupsi Umum (Non-Halal)')
       .replace('Disrupsi Rantai pasok halal', 'Disrupsi Halal')
    for idx in pivot.index
]
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', ax=ax4,
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Jumlah Artikel'})
ax4.set_title('🗓️ Heatmap Artikel: Label × Tahun', fontsize=12, fontweight='bold')
ax4.set_xlabel('Tahun', fontsize=11)
ax4.set_ylabel('', fontsize=11)
ax4.tick_params(axis='x', rotation=0)
ax4.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.savefig('panel4_heatmap.png', bbox_inches='tight', dpi=150)
plt.close()
print('✅ Tersimpan: panel4_heatmap.png')

---
# 🗺️ BAGIAN 3 — Peta Choropleth Indonesia

Visualisasi peta distribusi kejadian disrupsi supply chain di seluruh Indonesia, dari level provinsi hingga kabupaten/kota.

## 🌏 Peta Choropleth — Full 38 Provinsi

Menampilkan peta Indonesia tingkat provinsi dengan warna intensitas berdasarkan jumlah kejadian disrupsi. Menggunakan GeoJSON dari file lokal.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# PETA CHOROPLETH INDONESIA — FULL 38 PROVINSI, ACEH s.d. PAPUA BARAT DAYA
# ═════════════════════════════════════════════════════════════════════════════


# ─────────────────────────────────────────────────────────────────────────────
# 1.  MAPPING LENGKAP: kota/kabupaten  →  provinsi (normalisasi lowercase)
# ─────────────────────────────────────────────────────────────────────────────
KOTA_KE_PROVINSI: dict[str, str] = {

    # ── 1. ACEH ──────────────────────────────────────────────────────────────
    'banda aceh'      : 'aceh', 'langsa'          : 'aceh',
    'lhokseumawe'     : 'aceh', 'sabang'           : 'aceh',
    'subulussalam'    : 'aceh', 'aceh besar'       : 'aceh',
    'aceh barat'      : 'aceh', 'aceh barat daya'  : 'aceh',
    'aceh jaya'       : 'aceh', 'aceh selatan'     : 'aceh',
    'aceh singkil'    : 'aceh', 'singkil'          : 'aceh',
    'aceh tamiang'    : 'aceh', 'aceh tengah'      : 'aceh',
    'aceh tenggara'   : 'aceh', 'aceh timur'       : 'aceh',
    'aceh utara'      : 'aceh', 'bener meriah'     : 'aceh',
    'bireuen'         : 'aceh', 'gayo lues'        : 'aceh',
    'nagan raya'      : 'aceh', 'pidie'            : 'aceh',
    'pidie jaya'      : 'aceh', 'simeulue'         : 'aceh',
    'meulaboh'        : 'aceh', 'sigli'            : 'aceh',
    'takengon'        : 'aceh', 'blangkejeren'     : 'aceh',
    'tapaktuan'       : 'aceh',

    # ── 2. SUMATERA UTARA ─────────────────────────────────────────────────
    'medan'             : 'sumatera utara', 'binjai'            : 'sumatera utara',
    'sibolga'           : 'sumatera utara', 'tanjungbalai'      : 'sumatera utara',
    'pematangsiantar'   : 'sumatera utara', 'tebing tinggi'     : 'sumatera utara',
    'padangsidimpuan'   : 'sumatera utara', 'gunungsitoli'      : 'sumatera utara',
    'deli serdang'      : 'sumatera utara', 'langkat'           : 'sumatera utara',
    'karo'              : 'sumatera utara', 'simalungun'        : 'sumatera utara',
    'dairi'             : 'sumatera utara', 'tapanuli utara'    : 'sumatera utara',
    'tapanuli selatan'  : 'sumatera utara', 'tapanuli tengah'   : 'sumatera utara',
    'nias'              : 'sumatera utara', 'nias selatan'      : 'sumatera utara',
    'nias utara'        : 'sumatera utara', 'nias barat'        : 'sumatera utara',
    'mandailing natal'  : 'sumatera utara', 'labuhanbatu'       : 'sumatera utara',
    'asahan'            : 'sumatera utara', 'batu bara'         : 'sumatera utara',
    'serdang bedagai'   : 'sumatera utara', 'padang lawas'      : 'sumatera utara',
    'padang lawas utara': 'sumatera utara', 'humbang hasundutan': 'sumatera utara',
    'toba samosir'      : 'sumatera utara', 'samosir'           : 'sumatera utara',
    'pakpak bharat'     : 'sumatera utara', 'labuhanbatu utara' : 'sumatera utara',
    'labuhanbatu selatan':'sumatera utara',

    # ── 3. SUMATERA BARAT ─────────────────────────────────────────────────
    'padang'              : 'sumatera barat', 'bukittinggi'         : 'sumatera barat',
    'payakumbuh'          : 'sumatera barat', 'solok'               : 'sumatera barat',
    'sawahlunto'          : 'sumatera barat', 'padangpanjang'       : 'sumatera barat',
    'pariaman'            : 'sumatera barat', 'agam'                : 'sumatera barat',
    'tanah datar'         : 'sumatera barat', 'lima puluh kota'     : 'sumatera barat',
    'limapuluh kota'      : 'sumatera barat', 'pasaman'             : 'sumatera barat',
    'pasaman barat'       : 'sumatera barat', 'sijunjung'           : 'sumatera barat',
    'dharmasraya'         : 'sumatera barat', 'solok selatan'       : 'sumatera barat',
    'pesisir selatan'     : 'sumatera barat', 'kepulauan mentawai'  : 'sumatera barat',
    'padang pariaman'     : 'sumatera barat',

    # ── 4. RIAU ──────────────────────────────────────────────────────────
    'pekanbaru'        : 'riau', 'dumai'             : 'riau',
    'bengkalis'        : 'riau', 'siak'              : 'riau',
    'kampar'           : 'riau', 'rokan hulu'        : 'riau',
    'rokan hilir'      : 'riau', 'indragiri hulu'    : 'riau',
    'indragiri hilir'  : 'riau', 'pelalawan'         : 'riau',
    'kuantan singingi' : 'riau', 'kepulauan meranti' : 'riau',

    # ── 5. KEPULAUAN RIAU ────────────────────────────────────────────────
    'tanjungpinang'    : 'kepulauan riau', 'tanjung pinang'   : 'kepulauan riau',
    'batam'            : 'kepulauan riau', 'bintan'           : 'kepulauan riau',
    'karimun'          : 'kepulauan riau', 'natuna'           : 'kepulauan riau',
    'anambas'          : 'kepulauan riau', 'lingga'           : 'kepulauan riau',
    'kepulauan anambas': 'kepulauan riau',

    # ── 6. JAMBI ─────────────────────────────────────────────────────────
    'jambi'                  : 'jambi', 'sungai penuh'           : 'jambi',
    'muaro jambi'            : 'jambi', 'batanghari'             : 'jambi',
    'bungo'                  : 'jambi', 'tebo'                   : 'jambi',
    'merangin'               : 'jambi', 'sarolangun'             : 'jambi',
    'kerinci'                : 'jambi', 'tanjung jabung barat'   : 'jambi',
    'tanjung jabung timur'   : 'jambi',

    # ── 7. SUMATERA SELATAN ───────────────────────────────────────────────
    'palembang'                  : 'sumatera selatan',
    'prabumulih'                 : 'sumatera selatan',
    'pagaralam'                  : 'sumatera selatan',
    'pagar alam'                 : 'sumatera selatan',
    'lubuklinggau'               : 'sumatera selatan',
    'lahat'                      : 'sumatera selatan',
    'muara enim'                 : 'sumatera selatan',
    'ogan komering ilir'         : 'sumatera selatan',
    'ogan komering ulu'          : 'sumatera selatan',
    'musi banyuasin'             : 'sumatera selatan',
    'musi rawas'                 : 'sumatera selatan',
    'musi rawas utara'           : 'sumatera selatan',
    'empat lawang'               : 'sumatera selatan',
    'banyuasin'                  : 'sumatera selatan',
    'penukal abab'               : 'sumatera selatan',
    'penukal abab lematang ilir' : 'sumatera selatan',
    'ogan ilir'                  : 'sumatera selatan',
    'ogan komering ulu timur'    : 'sumatera selatan',
    'ogan komering ulu selatan'  : 'sumatera selatan',

    # ── 8. BANGKA BELITUNG ────────────────────────────────────────────────
    'pangkalpinang'  : 'bangka belitung', 'pangkal pinang'  : 'bangka belitung',
    'bangka'         : 'bangka belitung', 'belitung'        : 'bangka belitung',
    'bangka barat'   : 'bangka belitung', 'bangka tengah'   : 'bangka belitung',
    'bangka selatan' : 'bangka belitung', 'belitung timur'  : 'bangka belitung',

    # ── 9. BENGKULU ──────────────────────────────────────────────────────
    'bengkulu'         : 'bengkulu', 'rejang lebong'    : 'bengkulu',
    'kepahiang'        : 'bengkulu', 'lebong'           : 'bengkulu',
    'bengkulu utara'   : 'bengkulu', 'bengkulu tengah'  : 'bengkulu',
    'bengkulu selatan' : 'bengkulu', 'seluma'           : 'bengkulu',
    'kaur'             : 'bengkulu', 'muko muko'        : 'bengkulu',
    'mukomuko'         : 'bengkulu',

    # ── 10. LAMPUNG ──────────────────────────────────────────────────────
    'bandar lampung'     : 'lampung', 'metro'              : 'lampung',
    'lampung utara'      : 'lampung', 'lampung selatan'    : 'lampung',
    'lampung tengah'     : 'lampung', 'lampung barat'      : 'lampung',
    'lampung timur'      : 'lampung', 'mesuji'             : 'lampung',
    'tulang bawang'      : 'lampung', 'tulang bawang barat': 'lampung',
    'pesawaran'          : 'lampung', 'pringsewu'          : 'lampung',
    'tanggamus'          : 'lampung', 'way kanan'          : 'lampung',
    'pesisir barat'      : 'lampung',

    # ── 11. BANTEN ────────────────────────────────────────────────────────
    'serang'            : 'banten', 'cilegon'          : 'banten',
    'tangerang'         : 'banten', 'tangerang selatan': 'banten',
    'lebak'             : 'banten', 'pandeglang'       : 'banten',

    # ── 12. DKI JAKARTA ───────────────────────────────────────────────────
    'jakarta'          : 'dki jakarta', 'jakarta pusat'    : 'dki jakarta',
    'jakarta utara'    : 'dki jakarta', 'jakarta barat'    : 'dki jakarta',
    'jakarta selatan'  : 'dki jakarta', 'jakarta timur'    : 'dki jakarta',
    'kepulauan seribu' : 'dki jakarta',

    # ── 13. JAWA BARAT ────────────────────────────────────────────────────
    'bandung'        : 'jawa barat', 'bekasi'         : 'jawa barat',
    'bogor'          : 'jawa barat', 'depok'          : 'jawa barat',
    'cimahi'         : 'jawa barat', 'sukabumi'       : 'jawa barat',
    'cirebon'        : 'jawa barat', 'tasikmalaya'    : 'jawa barat',
    'banjar'         : 'jawa barat', 'karawang'       : 'jawa barat',
    'cikarang'       : 'jawa barat', 'purwakarta'     : 'jawa barat',
    'subang'         : 'jawa barat', 'garut'          : 'jawa barat',
    'cianjur'        : 'jawa barat', 'kuningan'       : 'jawa barat',
    'majalengka'     : 'jawa barat', 'indramayu'      : 'jawa barat',
    'sumedang'       : 'jawa barat', 'pangandaran'    : 'jawa barat',
    'bandung barat'  : 'jawa barat', 'ciamis'         : 'jawa barat',
    'cikampek'       : 'jawa barat',

    # ── 14. JAWA TENGAH ───────────────────────────────────────────────────
    'semarang'     : 'jawa tengah', 'surakarta'    : 'jawa tengah',
    'solo'         : 'jawa tengah', 'salatiga'     : 'jawa tengah',
    'magelang'     : 'jawa tengah', 'pekalongan'   : 'jawa tengah',
    'tegal'        : 'jawa tengah', 'purwokerto'   : 'jawa tengah',
    'cilacap'      : 'jawa tengah', 'banyumas'     : 'jawa tengah',
    'kebumen'      : 'jawa tengah', 'purworejo'    : 'jawa tengah',
    'wonosobo'     : 'jawa tengah', 'temanggung'   : 'jawa tengah',
    'kendal'       : 'jawa tengah', 'demak'        : 'jawa tengah',
    'jepara'       : 'jawa tengah', 'kudus'        : 'jawa tengah',
    'pati'         : 'jawa tengah', 'rembang'      : 'jawa tengah',
    'blora'        : 'jawa tengah', 'grobogan'     : 'jawa tengah',
    'sragen'       : 'jawa tengah', 'karanganyar'  : 'jawa tengah',
    'wonogiri'     : 'jawa tengah', 'sukoharjo'    : 'jawa tengah',
    'boyolali'     : 'jawa tengah', 'klaten'       : 'jawa tengah',
    'brebes'       : 'jawa tengah', 'pemalang'     : 'jawa tengah',
    'batang'       : 'jawa tengah', 'banjarnegara' : 'jawa tengah',
    'purbalingga'  : 'jawa tengah', 'gombong'      : 'jawa tengah',

    # ── 15. DI YOGYAKARTA ─────────────────────────────────────────────────
    'yogyakarta'  : 'daerah istimewa yogyakarta',
    'sleman'      : 'daerah istimewa yogyakarta',
    'bantul'      : 'daerah istimewa yogyakarta',
    'kulonprogo'  : 'daerah istimewa yogyakarta',
    'kulon progo' : 'daerah istimewa yogyakarta',
    'gunung kidul': 'daerah istimewa yogyakarta',
    'gunungkidul' : 'daerah istimewa yogyakarta',
    'wates'       : 'daerah istimewa yogyakarta',

    # ── 16. JAWA TIMUR ────────────────────────────────────────────────────
    'surabaya'    : 'jawa timur', 'malang'      : 'jawa timur',
    'kediri'      : 'jawa timur', 'blitar'      : 'jawa timur',
    'madiun'      : 'jawa timur', 'mojokerto'   : 'jawa timur',
    'pasuruan'    : 'jawa timur', 'probolinggo' : 'jawa timur',
    'batu'        : 'jawa timur', 'jember'      : 'jawa timur',
    'banyuwangi'  : 'jawa timur', 'situbondo'   : 'jawa timur',
    'bondowoso'   : 'jawa timur', 'lumajang'    : 'jawa timur',
    'jombang'     : 'jawa timur', 'nganjuk'     : 'jawa timur',
    'tulungagung' : 'jawa timur', 'trenggalek'  : 'jawa timur',
    'pacitan'     : 'jawa timur', 'ponorogo'    : 'jawa timur',
    'magetan'     : 'jawa timur', 'ngawi'       : 'jawa timur',
    'bojonegoro'  : 'jawa timur', 'lamongan'    : 'jawa timur',
    'gresik'      : 'jawa timur', 'sidoarjo'    : 'jawa timur',
    'tuban'       : 'jawa timur', 'bangkalan'   : 'jawa timur',
    'sampang'     : 'jawa timur', 'pamekasan'   : 'jawa timur',
    'sumenep'     : 'jawa timur',

    # ── 17. BALI ──────────────────────────────────────────────────────────
    'denpasar'    : 'bali', 'badung'      : 'bali',
    'gianyar'     : 'bali', 'tabanan'     : 'bali',
    'klungkung'   : 'bali', 'karangasem'  : 'bali',
    'buleleng'    : 'bali', 'jembrana'    : 'bali',
    'bangli'      : 'bali', 'kuta'        : 'bali',
    'ubud'        : 'bali', 'sanur'       : 'bali',
    'nusa dua'    : 'bali', 'singaraja'   : 'bali',

    # ── 18. NUSA TENGGARA BARAT ───────────────────────────────────────────
    'mataram'       : 'nusa tenggara barat', 'bima'          : 'nusa tenggara barat',
    'dompu'         : 'nusa tenggara barat', 'sumbawa'       : 'nusa tenggara barat',
    'sumbawa besar' : 'nusa tenggara barat', 'lombok'        : 'nusa tenggara barat',
    'lombok barat'  : 'nusa tenggara barat', 'lombok tengah' : 'nusa tenggara barat',
    'lombok timur'  : 'nusa tenggara barat', 'lombok utara'  : 'nusa tenggara barat',
    'sumbawa barat' : 'nusa tenggara barat', 'raba'          : 'nusa tenggara barat',

    # ── 19. NUSA TENGGARA TIMUR ───────────────────────────────────────────
    'kupang'              : 'nusa tenggara timur', 'ende'                : 'nusa tenggara timur',
    'maumere'             : 'nusa tenggara timur', 'bajawa'              : 'nusa tenggara timur',
    'ruteng'              : 'nusa tenggara timur', 'labuan bajo'         : 'nusa tenggara timur',
    'waingapu'            : 'nusa tenggara timur', 'waikabubak'          : 'nusa tenggara timur',
    'atambua'             : 'nusa tenggara timur', 'kefamenanu'          : 'nusa tenggara timur',
    'soe'                 : 'nusa tenggara timur', 'manggarai'           : 'nusa tenggara timur',
    'manggarai barat'     : 'nusa tenggara timur', 'manggarai timur'     : 'nusa tenggara timur',
    'nagekeo'             : 'nusa tenggara timur', 'ngada'               : 'nusa tenggara timur',
    'sikka'               : 'nusa tenggara timur', 'alor'                : 'nusa tenggara timur',
    'lembata'             : 'nusa tenggara timur', 'flores timur'        : 'nusa tenggara timur',
    'rote ndao'           : 'nusa tenggara timur', 'sumba barat'         : 'nusa tenggara timur',
    'sumba timur'         : 'nusa tenggara timur', 'sumba tengah'        : 'nusa tenggara timur',
    'sumba barat daya'    : 'nusa tenggara timur', 'timor tengah utara'  : 'nusa tenggara timur',
    'timor tengah selatan': 'nusa tenggara timur', 'belu'                : 'nusa tenggara timur',
    'malaka'              : 'nusa tenggara timur', 'sabu raijua'         : 'nusa tenggara timur',

    # ── 20. KALIMANTAN BARAT ─────────────────────────────────────────────
    'pontianak'   : 'kalimantan barat', 'singkawang'  : 'kalimantan barat',
    'sambas'      : 'kalimantan barat', 'mempawah'    : 'kalimantan barat',
    'bengkayang'  : 'kalimantan barat', 'landak'      : 'kalimantan barat',
    'sanggau'     : 'kalimantan barat', 'sekadau'     : 'kalimantan barat',
    'sintang'     : 'kalimantan barat', 'melawi'      : 'kalimantan barat',
    'kapuas hulu' : 'kalimantan barat', 'ketapang'    : 'kalimantan barat',
    'kayong utara': 'kalimantan barat', 'kubu raya'   : 'kalimantan barat',

    # ── 21. KALIMANTAN TENGAH ─────────────────────────────────────────────
    'palangka raya'      : 'kalimantan tengah', 'sampit'             : 'kalimantan tengah',
    'pangkalan bun'      : 'kalimantan tengah', 'muara teweh'        : 'kalimantan tengah',
    'kuala kapuas'       : 'kalimantan tengah', 'buntok'             : 'kalimantan tengah',
    'tamiang layang'     : 'kalimantan tengah', 'kotawaringin barat' : 'kalimantan tengah',
    'kotawaringin timur' : 'kalimantan tengah', 'barito utara'       : 'kalimantan tengah',
    'barito selatan'     : 'kalimantan tengah', 'barito timur'       : 'kalimantan tengah',
    'pulang pisau'       : 'kalimantan tengah', 'gunung mas'         : 'kalimantan tengah',
    'katingan'           : 'kalimantan tengah', 'lamandau'           : 'kalimantan tengah',
    'seruyan'            : 'kalimantan tengah', 'sukamara'           : 'kalimantan tengah',
    'murung raya'        : 'kalimantan tengah', 'kapuas'             : 'kalimantan tengah',

    # ── 22. KALIMANTAN SELATAN ────────────────────────────────────────────
    'banjarmasin'       : 'kalimantan selatan', 'banjarbaru'        : 'kalimantan selatan',
    'martapura'         : 'kalimantan selatan', 'amuntai'           : 'kalimantan selatan',
    'kandangan'         : 'kalimantan selatan', 'pelaihari'         : 'kalimantan selatan',
    'batulicin'         : 'kalimantan selatan', 'kotabaru'          : 'kalimantan selatan',
    'tanjung'           : 'kalimantan selatan', 'rantau'            : 'kalimantan selatan',
    'barito kuala'      : 'kalimantan selatan', 'hulu sungai utara' : 'kalimantan selatan',
    'hulu sungai tengah': 'kalimantan selatan', 'hulu sungai selatan':'kalimantan selatan',
    'tabalong'          : 'kalimantan selatan', 'tapin'             : 'kalimantan selatan',
    'tanah laut'        : 'kalimantan selatan', 'tanah bumbu'       : 'kalimantan selatan',
    'balangan'          : 'kalimantan selatan', 'banjar kab'        : 'kalimantan selatan',

    # ── 23. KALIMANTAN TIMUR ─────────────────────────────────────────────
    'samarinda'          : 'kalimantan timur', 'balikpapan'         : 'kalimantan timur',
    'bontang'            : 'kalimantan timur', 'tarakan'            : 'kalimantan timur',
    'tenggarong'         : 'kalimantan timur', 'sangatta'           : 'kalimantan timur',
    'kutai kartanegara'  : 'kalimantan timur', 'kutai barat'        : 'kalimantan timur',
    'kutai timur'        : 'kalimantan timur', 'berau'              : 'kalimantan timur',
    'penajam paser utara': 'kalimantan timur', 'paser'              : 'kalimantan timur',
    'mahakam ulu'        : 'kalimantan timur',

    # ── 24. KALIMANTAN UTARA ─────────────────────────────────────────────
    'tanjung selor' : 'kalimantan utara', 'nunukan'       : 'kalimantan utara',
    'malinau'       : 'kalimantan utara', 'bulungan'      : 'kalimantan utara',
    'tana tidung'   : 'kalimantan utara',

    # ── 25. SULAWESI UTARA ────────────────────────────────────────────────
    'manado'                    : 'sulawesi utara', 'bitung'                    : 'sulawesi utara',
    'tomohon'                   : 'sulawesi utara', 'kotamobagu'                : 'sulawesi utara',
    'minahasa'                  : 'sulawesi utara', 'minahasa utara'            : 'sulawesi utara',
    'minahasa selatan'          : 'sulawesi utara', 'minahasa tenggara'         : 'sulawesi utara',
    'bolaang mongondow'         : 'sulawesi utara', 'bolaang mongondow utara'   : 'sulawesi utara',
    'bolaang mongondow timur'   : 'sulawesi utara', 'bolaang mongondow selatan' : 'sulawesi utara',
    'sangihe'                   : 'sulawesi utara', 'kepulauan sangihe'         : 'sulawesi utara',
    'sitaro'                    : 'sulawesi utara', 'kepulauan siau tagulandang biaro':'sulawesi utara',
    'talaud'                    : 'sulawesi utara', 'kepulauan talaud'          : 'sulawesi utara',

    # ── 26. SULAWESI TENGAH ───────────────────────────────────────────────
    'palu'            : 'sulawesi tengah', 'luwuk'          : 'sulawesi tengah',
    'toli toli'       : 'sulawesi tengah', 'toli-toli'      : 'sulawesi tengah',
    'buol'            : 'sulawesi tengah', 'donggala'       : 'sulawesi tengah',
    'sigi'            : 'sulawesi tengah', 'parigi moutong' : 'sulawesi tengah',
    'poso'            : 'sulawesi tengah', 'tojo una una'   : 'sulawesi tengah',
    'tojo una-una'    : 'sulawesi tengah', 'banggai'        : 'sulawesi tengah',
    'banggai kepulauan':'sulawesi tengah', 'morowali'       : 'sulawesi tengah',
    'morowali utara'  : 'sulawesi tengah', 'banggai laut'   : 'sulawesi tengah',

    # ── 27. SULAWESI SELATAN ──────────────────────────────────────────────
    'makassar'                    : 'sulawesi selatan', 'parepare'                  : 'sulawesi selatan',
    'palopo'                      : 'sulawesi selatan', 'maros'                     : 'sulawesi selatan',
    'gowa'                        : 'sulawesi selatan', 'takalar'                   : 'sulawesi selatan',
    'jeneponto'                   : 'sulawesi selatan', 'bantaeng'                  : 'sulawesi selatan',
    'bulukumba'                   : 'sulawesi selatan', 'sinjai'                    : 'sulawesi selatan',
    'bone'                        : 'sulawesi selatan', 'soppeng'                   : 'sulawesi selatan',
    'wajo'                        : 'sulawesi selatan', 'sidrap'                    : 'sulawesi selatan',
    'sidenreng rappang'           : 'sulawesi selatan', 'pinrang'                   : 'sulawesi selatan',
    'enrekang'                    : 'sulawesi selatan', 'tana toraja'               : 'sulawesi selatan',
    'toraja utara'                : 'sulawesi selatan', 'luwu'                      : 'sulawesi selatan',
    'luwu utara'                  : 'sulawesi selatan', 'luwu timur'                : 'sulawesi selatan',
    'kepulauan selayar'           : 'sulawesi selatan', 'selayar'                   : 'sulawesi selatan',
    'barru'                       : 'sulawesi selatan', 'pangkep'                   : 'sulawesi selatan',
    'pangkajene'                  : 'sulawesi selatan', 'pangkajene dan kepulauan'  : 'sulawesi selatan',

    # ── 28. SULAWESI TENGGARA ─────────────────────────────────────────────
    'kendari'           : 'sulawesi tenggara', 'baubau'             : 'sulawesi tenggara',
    'bau-bau'           : 'sulawesi tenggara', 'kolaka'             : 'sulawesi tenggara',
    'kolaka utara'      : 'sulawesi tenggara', 'kolaka timur'       : 'sulawesi tenggara',
    'konawe'            : 'sulawesi tenggara', 'konawe selatan'     : 'sulawesi tenggara',
    'konawe utara'      : 'sulawesi tenggara', 'konawe kepulauan'   : 'sulawesi tenggara',
    'muna'              : 'sulawesi tenggara', 'muna barat'         : 'sulawesi tenggara',
    'buton'             : 'sulawesi tenggara', 'buton utara'        : 'sulawesi tenggara',
    'buton selatan'     : 'sulawesi tenggara', 'buton tengah'       : 'sulawesi tenggara',
    'bombana'           : 'sulawesi tenggara', 'wakatobi'           : 'sulawesi tenggara',
    'raha'              : 'sulawesi tenggara',

    # ── 29. GORONTALO ─────────────────────────────────────────────────────
    'gorontalo'       : 'gorontalo', 'gorontalo utara' : 'gorontalo',
    'bone bolango'    : 'gorontalo', 'boalemo'         : 'gorontalo',
    'pohuwato'        : 'gorontalo',

    # ── 30. SULAWESI BARAT ────────────────────────────────────────────────
    'mamuju'          : 'sulawesi barat', 'majene'          : 'sulawesi barat',
    'polewali mandar' : 'sulawesi barat', 'mamasa'          : 'sulawesi barat',
    'mamuju utara'    : 'sulawesi barat', 'pasangkayu'      : 'sulawesi barat',
    'mamuju tengah'   : 'sulawesi barat',

    # ── 31. MALUKU ────────────────────────────────────────────────────────
    'ambon'               : 'maluku', 'tual'                : 'maluku',
    'masohi'              : 'maluku', 'namlea'              : 'maluku',
    'bula'                : 'maluku', 'dobo'                : 'maluku',
    'saumlaki'            : 'maluku', 'maluku tengah'       : 'maluku',
    'maluku tenggara'     : 'maluku', 'maluku barat daya'   : 'maluku',
    'seram bagian barat'  : 'maluku', 'seram bagian timur'  : 'maluku',
    'kepulauan aru'       : 'maluku', 'buru'                : 'maluku',
    'buru selatan'        : 'maluku', 'maluku tenggara barat': 'maluku',

    # ── 32. MALUKU UTARA ─────────────────────────────────────────────────
    'ternate'            : 'maluku utara', 'tidore'             : 'maluku utara',
    'tidore kepulauan'   : 'maluku utara', 'tobelo'             : 'maluku utara',
    'labuha'             : 'maluku utara', 'sanana'             : 'maluku utara',
    'sofifi'             : 'maluku utara', 'halmahera utara'    : 'maluku utara',
    'halmahera selatan'  : 'maluku utara', 'halmahera barat'    : 'maluku utara',
    'halmahera timur'    : 'maluku utara', 'halmahera tengah'   : 'maluku utara',
    'kepulauan sula'     : 'maluku utara', 'pulau morotai'      : 'maluku utara',
    'pulau taliabu'      : 'maluku utara',

    # ── 33. PAPUA ─────────────────────────────────────────────────────────
    'jayapura'         : 'papua', 'sentani'          : 'papua',
    'abepura'          : 'papua', 'sarmi'            : 'papua',
    'keerom'           : 'papua', 'waropen'          : 'papua',
    'supiori'          : 'papua', 'mamberamo raya'   : 'papua',
    'biak numfor'      : 'papua', 'kepulauan yapen'  : 'papua',
    'yahukimo'         : 'papua',

    # ── 34. PAPUA BARAT ───────────────────────────────────────────────────
    'manokwari'          : 'papua barat', 'fakfak'             : 'papua barat',
    'kaimana'            : 'papua barat', 'teluk bintuni'      : 'papua barat',
    'teluk wondama'      : 'papua barat', 'manokwari selatan'  : 'papua barat',
    'pegunungan arfak'   : 'papua barat', 'tambrauw'           : 'papua barat',
    'maybrat'            : 'papua barat',

    # ── 35. PAPUA SELATAN (provinsi baru 2022) ────────────────────────────
    'merauke'      : 'papua selatan', 'boven digoel'  : 'papua selatan',
    'mappi'        : 'papua selatan', 'asmat'         : 'papua selatan',

    # ── 36. PAPUA TENGAH (provinsi baru 2022) ────────────────────────────
    'nabire'       : 'papua tengah', 'paniai'       : 'papua tengah',
    'dogiyai'      : 'papua tengah', 'intan jaya'   : 'papua tengah',
    'deiyai'       : 'papua tengah', 'timika'       : 'papua tengah',
    'mimika'       : 'papua tengah', 'puncak jaya'  : 'papua tengah',
    'puncak'       : 'papua tengah',

    # ── 37. PAPUA PEGUNUNGAN (provinsi baru 2022) ─────────────────────────
    'wamena'              : 'papua pegunungan', 'jayawijaya'          : 'papua pegunungan',
    'pegunungan bintang'  : 'papua pegunungan', 'tolikara'            : 'papua pegunungan',
    'nduga'               : 'papua pegunungan', 'lanny jaya'          : 'papua pegunungan',
    'mamberamo tengah'    : 'papua pegunungan', 'yalimo'              : 'papua pegunungan',

    # ── 38. PAPUA BARAT DAYA (provinsi baru 2022) ─────────────────────────
    'sorong'         : 'papua barat daya', 'sorong selatan'   : 'papua barat daya',
    'raja ampat'     : 'papua barat daya',
}

# Tambahkan juga nama provinsi itu sendiri sebagai self-mapping
for prov in [
    'aceh','sumatera utara','sumatera barat','riau','kepulauan riau',
    'jambi','sumatera selatan','bangka belitung','bengkulu','lampung',
    'dki jakarta','banten','jawa barat','jawa tengah',
    'daerah istimewa yogyakarta','jawa timur','bali',
    'nusa tenggara barat','nusa tenggara timur',
    'kalimantan barat','kalimantan tengah','kalimantan selatan',
    'kalimantan timur','kalimantan utara',
    'sulawesi utara','sulawesi tengah','sulawesi selatan',
    'sulawesi tenggara','gorontalo','sulawesi barat',
    'maluku','maluku utara',
    'papua','papua barat','papua selatan','papua tengah',
    'papua pegunungan','papua barat daya',
]:
    KOTA_KE_PROVINSI.setdefault(prov, prov)

print(f'✅ Total entri mapping: {len(KOTA_KE_PROVINSI)}')


# ─────────────────────────────────────────────────────────────────────────────
# 2.  AGREGASI KOTA → PROVINSI
# ─────────────────────────────────────────────────────────────────────────────
def agregasi_ke_provinsi(counter_kota: dict) -> dict:
    hasil = {}
    tidak_terpetakan = []
    for kota, freq in counter_kota.items():
        prov = KOTA_KE_PROVINSI.get(kota.lower().strip())
        if prov:
            hasil[prov] = hasil.get(prov, 0) + freq
        else:
            tidak_terpetakan.append((kota, freq))
    if tidak_terpetakan:
        print(f'⚠️  {len(tidak_terpetakan)} kota tidak terpetakan:')
        for k, f in sorted(tidak_terpetakan, key=lambda x: -x[1])[:20]:
            print(f'   - "{k}" (freq={f})')
    return hasil

prov_freq_all          = agregasi_ke_provinsi(dict(counter_all))
prov_freq_disrupsi     = agregasi_ke_provinsi(dict(counter_disrupsi))
prov_freq_non_disrupsi = agregasi_ke_provinsi(dict(counter_non_disrupsi))

print('\n📊 Top-20 provinsi (semua artikel):')
for k, v in sorted(prov_freq_all.items(), key=lambda x: -x[1])[:20]:
    print(f'   {k:<35} {v}')


# ─────────────────────────────────────────────────────────────────────────────
# 3.  DOWNLOAD & LOAD GEOJSON INDONESIA (LEVEL PROVINSI)
#     ✅ FIX: Iterasi tiap URL satu per satu, validasi JSON sebelum simpan
# ─────────────────────────────────────────────────────────────────────────────
GEOJSON_MIRRORS = [
    'https://raw.githubusercontent.com/superpikar/indonesia-geojson/'
    'master/indonesia-provinces-simple.json',
    'https://raw.githubusercontent.com/datasets/geo-admin1-indonesia/'
    'main/data/geo-admin1-indonesia.geojson',
    'https://raw.githubusercontent.com/wahyu-adi-n/geojson-indonesia/'
    'main/provinces.geojson',
    'https://raw.githubusercontent.com/kodewilayah/permendagri-72-2019/'
    'main/dist/province.json',
    'https://raw.githubusercontent.com/indietl/indonesia-maps/'
    'master/indonesia-geojson/indonesia-provinces.geojson',
]

GEOJSON_PATH = 'indonesia_provinces.geojson'

def download_geojson(path: str, mirrors: list[str]) -> bool:
    """
    Coba unduh GeoJSON dari daftar mirror satu per satu.
    Validasi respons sebagai JSON yang valid sebelum disimpan.
    Kembalikan True jika berhasil.
    """
    for url in mirrors:
        try:
            print(f'⬇️  Mencoba: {url}')
            r = requests.get(url, timeout=30)
            r.raise_for_status()                  # error jika status 4xx/5xx
            # ── Validasi: pastikan isi adalah JSON yang bisa di-parse ──────
            data = r.json()                        # akan raise jika bukan JSON
            # ── Validasi ringan: harus punya key 'features' ─────────────────
            if 'features' not in data:
                print(f'   ⚠️  Tidak ada key "features", skip.')
                continue
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f)
            print(f'✅ GeoJSON berhasil diunduh dari: {url}')
            return True
        except Exception as e:
            print(f'   ✗ Gagal: {e}')
    return False

# Hapus file lama jika korup (ukuran < 10 KB kemungkinan besar salah)
if os.path.exists(GEOJSON_PATH):
    if os.path.getsize(GEOJSON_PATH) < 10_000:
        print('⚠️  File GeoJSON terdeteksi korup (terlalu kecil), menghapus...')
        os.remove(GEOJSON_PATH)
    else:
        # Cek apakah file bisa di-parse sebagai JSON valid
        try:
            with open(GEOJSON_PATH, 'r', encoding='utf-8') as f:
                _test = json.load(f)
            if 'features' not in _test:
                raise ValueError('Tidak ada key "features"')
            print('✅ File GeoJSON lokal valid, skip download.')
        except Exception as e:
            print(f'⚠️  File GeoJSON lokal rusak ({e}), akan diunduh ulang.')
            os.remove(GEOJSON_PATH)

if not os.path.exists(GEOJSON_PATH):
    ok = download_geojson(GEOJSON_PATH, GEOJSON_MIRRORS)
    if not ok:
        raise RuntimeError(
            '❌ Semua mirror gagal. Pastikan koneksi internet aktif, '
            'atau letakkan file GeoJSON provinsi Indonesia secara manual '
            f'di path: {os.path.abspath(GEOJSON_PATH)}'
        )

# ── Load dengan GeoPandas ────────────────────────────────────────────────────
gdf = gpd.read_file(GEOJSON_PATH)

# Deteksi kolom nama provinsi (urutan prioritas)
NAME_COL = next(
    (c for c in ['Propinsi', 'name', 'NAME_1', 'NAME', 'provinsi', 'province']
     if c in gdf.columns),
    None,
)
if NAME_COL is None:
    raise ValueError(
        f'Kolom nama provinsi tidak ditemukan. Kolom tersedia: {gdf.columns.tolist()}'
    )
print(f'✅ Kolom nama: "{NAME_COL}" | Jumlah fitur: {len(gdf)}')

# ── Normalisasi & remap nama GeoJSON → kunci provinsi kita ──────────────────
GEOJSON_REMAP = {
    'di yogyakarta'            : 'daerah istimewa yogyakarta',
    'd.i. yogyakarta'          : 'daerah istimewa yogyakarta',
    'diy'                      : 'daerah istimewa yogyakarta',
    'di aceh'                  : 'aceh',
    'nanggroe aceh darussalam' : 'aceh',
    'nangroe aceh darussalam'  : 'aceh',
    'dki jakarta'              : 'dki jakarta',
    'jakarta raya'             : 'dki jakarta',
    'jakarta'                  : 'dki jakarta',
    'kepulauan bangka belitung': 'bangka belitung',
    'bangka-belitung'          : 'bangka belitung',
    'irian jaya barat'         : 'papua barat',
    'irian jaya'               : 'papua',
    'ntt'                      : 'nusa tenggara timur',
    'ntb'                      : 'nusa tenggara barat',
}

gdf['prov_key'] = (
    gdf[NAME_COL]
    .str.lower()
    .str.strip()
    .map(lambda x: GEOJSON_REMAP.get(x, x))
)

# ── Gabungkan frekuensi ──────────────────────────────────────────────────────
for col, freq_dict in [
    ('freq_all',          prov_freq_all),
    ('freq_disrupsi',     prov_freq_disrupsi),
    ('freq_non_disrupsi', prov_freq_non_disrupsi),
]:
    gdf[col] = gdf['prov_key'].map(freq_dict).fillna(0).astype(int)

print('\n✅ Cek penggabungan data:')
print(
    gdf[['prov_key', 'freq_all', 'freq_disrupsi', 'freq_non_disrupsi']]
    .sort_values('freq_all', ascending=False)
    .to_string(index=False)
)


# ─────────────────────────────────────────────────────────────────────────────
# 4.  FUNGSI PLOT CHOROPLETH — FULL INDONESIA
# ─────────────────────────────────────────────────────────────────────────────
def plot_choropleth_indonesia(
    gdf_map    : gpd.GeoDataFrame,
    col        : str,
    title      : str,
    filename   : str,
    cmap_name  : str   = 'RdYlGn_r',  # Merah=tinggi · Kuning=sedang · Hijau=rendah
    no_data_clr: str   = '#C8C8C8',   # Abu-abu muda → tidak ada data
    figsize    : tuple = (22, 11),
):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor('#F0F4F8')
    ax.set_facecolor('#D6EAF8')  # Warna laut

    vals = gdf_map[col]
    vmax = vals.max()

    if vmax == 0:
        gdf_map.plot(ax=ax, color=no_data_clr, edgecolor='white', linewidth=0.4)
    else:
        norm = mcolors.Normalize(vmin=0, vmax=vmax)
        cmap = cm.get_cmap(cmap_name)

        mask_data   = vals > 0
        mask_nodata = ~mask_data

        # Wilayah tanpa data
        if mask_nodata.any():
            gdf_map[mask_nodata].plot(
                ax=ax, color=no_data_clr,
                edgecolor='white', linewidth=0.4,
            )

        # Wilayah berdata — gradasi warna
        gdf_map[mask_data].plot(
            column=col, ax=ax,
            cmap=cmap_name, norm=norm,
            edgecolor='white', linewidth=0.4,
            legend=False,
        )

        # ── Colorbar ──────────────────────────────────────────────────────
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.018, pad=0.01,
                            aspect=30, shrink=0.6)
        cbar.set_label(
            'Frekuensi Kemunculan\nsebagai Lokasi Kejadian',
            fontsize=10, labelpad=8,
        )
        cbar.ax.tick_params(labelsize=8)
        for pos, lbl, clr in [
            (0.04, 'Rendah', '#2e7d32'),
            (0.50, 'Sedang', '#795548'),
            (0.94, 'Tinggi', '#b71c1c'),
        ]:
            cbar.ax.text(
                1.6, pos, lbl,
                transform=cbar.ax.transAxes,
                fontsize=8, color=clr, va='center',
            )

        # ── Label angka di centroid provinsi ──────────────────────────────
        for _, row in gdf_map[mask_data].iterrows():
            try:
                cx  = row.geometry.centroid.x
                cy  = row.geometry.centroid.y
                val = int(row[col])
                ratio   = val / vmax
                txt_clr = 'white' if ratio > 0.55 else '#1a1a1a'
                ax.text(
                    cx, cy, str(val),
                    fontsize=6.5, ha='center', va='center',
                    color=txt_clr, fontweight='bold',
                    bbox=dict(
                        boxstyle='round,pad=0.15',
                        fc='none', ec='none', alpha=0.6,
                    ),
                )
            except Exception:
                pass

    # ── Legend "tidak ada data" ────────────────────────────────────────────
    patch = mpatches.Patch(
        facecolor=no_data_clr, edgecolor='white',
        label='Tidak ada data',
    )
    ax.legend(handles=[patch], loc='lower left', fontsize=9,
              framealpha=0.85, edgecolor='#aaa')

    # ── Judul & layout ────────────────────────────────────────────────────
    ax.set_title(title, fontsize=14, fontweight='bold', pad=14, color='#1a1a2e')
    ax.set_axis_off()

    minx, miny, maxx, maxy = gdf_map.total_bounds
    margin_x = (maxx - minx) * 0.02
    margin_y = (maxy - miny) * 0.06
    ax.set_xlim(minx - margin_x, maxx + margin_x)
    ax.set_ylim(miny - margin_y, maxy + margin_y)

    plt.tight_layout()
    plt.savefig(filename, bbox_inches='tight', dpi=160)
    plt.show()
    print(f'✅ Tersimpan: {filename}')


# ─────────────────────────────────────────────────────────────────────────────
# 5.  RENDER 3 PETA FULL INDONESIA
# ─────────────────────────────────────────────────────────────────────────────
MAPS = [
    {
        'col'     : 'freq_all',
        'title'   : '🗺️  Peta Lokasi Kejadian Supply Chain — Semua Artikel (Aceh s.d. Papua)',
        'filename': 'peta_full_semua.png',
    },
    {
        'col'     : 'freq_disrupsi',
        'title'   : '🔴  Peta Lokasi Kejadian Supply Chain — Artikel Disrupsi',
        'filename': 'peta_full_disrupsi.png',
    },
    {
        'col'     : 'freq_non_disrupsi',
        'title'   : '🟢  Peta Lokasi Kejadian Supply Chain — Artikel Non-Disrupsi',
        'filename': 'peta_full_non_disrupsi.png',
    },
]

for m in MAPS:
    plot_choropleth_indonesia(gdf_map=gdf, **m)

print('\n✅ Semua peta berhasil dibuat!')
print('=' * 60)
print('📌 RINGKASAN ANALISIS SUPPLY CHAIN DISRUPSI')
print('=' * 60)

print('\n📊 Total Artikel per Label:')
print(df['Kategori_LLM'].value_counts().to_string())

print('\n🏙️ Top 10 Kota Paling Sering Disebut (Semua Artikel):')
print(df_kota_all.head(10).to_string(index=False))

print('\n🔴 Top 10 Kota dalam Artikel Disrupsi:')
print(df_kota_disrupsi.head(10).to_string(index=False))

print('\n📅 Tahun dengan Artikel Disrupsi Terbanyak:')
print(tahun_disrupsi.sort_values(ascending=False).to_string())

print('\n✅ Analisis selesai!')

## 🗺️ Peta Choropleth — Level Kabupaten/Kota

Peta resolusi tinggi per kabupaten/kota dengan label laut, blacklist komoditas, dan peta per pulau (Sumatera, Jawa, Kalimantan, Sulawesi, Bali & NTT, Maluku, Papua). Versi final dengan fix Sulawesi dan Bali/NTT.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# PETA CHOROPLETH INDONESIA — LEVEL KABUPATEN/KOTA
# Versi: paper-ready, label laut detail, blacklist komoditas, peta per pulau
# Fix: Sulawesi tidak terpotong, Bali & NTT muncul, label nama kota di peta pulau
# ═════════════════════════════════════════════════════════════════════════════



# ─────────────────────────────────────────────────────────────────────────────
# BLACKLIST: kata-kata yang bukan nama kota
# ─────────────────────────────────────────────────────────────────────────────
BLACKLIST: set = {
    'batu', 'batu kali', 'batu split', 'batu alam', 'batu andesit',
    'pasir', 'pasir besi', 'kerikil', 'tanah', 'tanah liat', 'semen',
    'besi', 'baja', 'aluminium', 'plastik', 'kayu', 'papan',
    'batu bara', 'batubara', 'coal', 'batu bara coking', 'batu bara thermal',
    'batu bara lignit', 'batu bara antrasit',
    'minyak', 'minyak bumi', 'minyak mentah', 'crude oil',
    'gas', 'gas alam', 'gas bumi', 'lng', 'lpg', 'bbm',
    'nikel', 'tembaga', 'emas', 'perak', 'bauksit', 'timah',
    'bijih besi', 'iron ore', 'feronikel', 'kobalt',
    'sawit', 'kelapa sawit', 'cpo', 'minyak sawit',
    'karet', 'kakao', 'kopi', 'teh', 'gula', 'tebu', 'jagung',
    'kedelai', 'beras', 'padi', 'gandum', 'singkong',
    'udang', 'ikan', 'tuna', 'kakap',
    'pelabuhan', 'bandara', 'gudang', 'depo', 'terminal',
    'tol', 'jalan tol', 'kereta', 'kapal', 'kontainer',
    'ekspor', 'impor', 'distribusi', 'pengiriman',
    'indonesia', 'nasional', 'pusat', 'regional', 'lokal',
}


# ─────────────────────────────────────────────────────────────────────────────
# UTILITAS NORMALISASI
# ─────────────────────────────────────────────────────────────────────────────
def normalize(s: str) -> str:
    s = s.lower().strip()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

_PREFIXES = [
    'kabupaten administrasi ', 'kota administrasi ',
    'kabupaten ', 'kota ', 'kab. ', 'kab ',
]

def strip_prefix(s: str) -> str:
    n = normalize(s)
    for p in _PREFIXES:
        if n.startswith(p):
            return n[len(p):]
    return n

def is_blacklisted(nama: str) -> bool:
    n = normalize(nama)
    return n in BLACKLIST or strip_prefix(nama) in BLACKLIST


# ─────────────────────────────────────────────────────────────────────────────
# DOWNLOAD GEOJSON KABUPATEN/KOTA
# ─────────────────────────────────────────────────────────────────────────────
KOTA_GEOJSON_MIRRORS = [
    'https://raw.githubusercontent.com/ans-4175/peta-indonesia-geojson/'
    'master/indonesia-kota.geojson',
    'https://raw.githubusercontent.com/riyaldirivai/geojson-indonesia/'
    'main/kabupaten_kota.geojson',
    'https://raw.githubusercontent.com/superpikar/indonesia-geojson/'
    'master/indonesia.json',
    'https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_IDN_2.json',
]
KOTA_GEOJSON_PATH = 'indonesia_kota.geojson'

def download_geojson(path, mirrors):
    for url in mirrors:
        try:
            print(f'⬇️  Mencoba: {url}')
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            data = r.json()
            if 'features' not in data or len(data['features']) == 0:
                print('   ⚠️  Tidak ada features, skip.')
                continue
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f)
            print(f'✅ Berhasil: {len(data["features"])} fitur')
            return True
        except Exception as e:
            print(f'   ✗ Gagal: {e}')
    return False

def _valid(path):
    if not os.path.exists(path) or os.path.getsize(path) < 50_000:
        return False
    try:
        with open(path, 'r', encoding='utf-8') as f:
            d = json.load(f)
        return 'features' in d and len(d['features']) > 100
    except Exception:
        return False

if not _valid(KOTA_GEOJSON_PATH):
    if os.path.exists(KOTA_GEOJSON_PATH):
        os.remove(KOTA_GEOJSON_PATH)
    if not download_geojson(KOTA_GEOJSON_PATH, KOTA_GEOJSON_MIRRORS):
        raise RuntimeError('❌ Semua mirror GeoJSON gagal.')

gdf = gpd.read_file(KOTA_GEOJSON_PATH)
print(f'✅ GeoJSON dimuat: {len(gdf)} fitur | Kolom: {gdf.columns.tolist()}')

NAME_CANDIDATES = ['NAME_2', 'name', 'NAME', 'kabupaten', 'kota',
                   'KABKOT', 'WADMKK', 'NAMOBJ', 'nama', 'daerah']
NAME_COL = next((c for c in NAME_CANDIDATES if c in gdf.columns), None)
if NAME_COL is None:
    raise ValueError(f'Kolom nama kota tidak ditemukan. Kolom: {gdf.columns.tolist()}')
print(f'   Kolom nama: "{NAME_COL}"')

gdf['geo_key']    = gdf[NAME_COL].astype(str).apply(normalize)
gdf['geo_key_np'] = gdf[NAME_COL].astype(str).apply(strip_prefix)

geo_keys_set    = set(gdf['geo_key'])
geo_keys_np_set = set(gdf['geo_key_np'])


# ─────────────────────────────────────────────────────────────────────────────
# ALIAS EJAAN
# ─────────────────────────────────────────────────────────────────────────────
ALIAS: dict = {
    'jakarta'        : 'jakarta pusat',
    'jakarta raya'   : 'jakarta pusat',
    'solo'           : 'surakarta',
    'jogja'          : 'yogyakarta',
    'tanjung pinang' : 'tanjungpinang',
    'limapuluh kota' : 'lima puluh kota',
    'labuan bajo'    : 'manggarai barat',
    'sentani'        : 'jayapura',
    'abepura'        : 'jayapura',
    'timika'         : 'mimika',
    'pangkep'        : 'pangkajene dan kepulauan',
    'pangkajene'     : 'pangkajene dan kepulauan',
    'bau-bau'        : 'baubau',
    'banjar kab'     : 'banjar',
    'singkil'        : 'aceh singkil',
    'meulaboh'       : 'aceh barat',
    'sigli'          : 'pidie',
    'takengon'       : 'aceh tengah',
    'blangkejeren'   : 'gayo lues',
    'tapaktuan'      : 'aceh selatan',
    'padangpanjang'  : 'padang panjang',
    'tanjungbalai'   : 'tanjung balai',
    'pematangsiantar': 'pematang siantar',
    'gunungsitoli'   : 'gunung sitoli',
    'padangsidimpuan': 'padang sidimpuan',
    'toli-toli'      : 'toli toli',
    'tojo una-una'   : 'tojo una una',
    'mamuju utara'   : 'pasangkayu',
    'tidore'         : 'tidore kepulauan',
}

def cari_geo_key(nama_kota):
    n   = normalize(nama_kota)
    np_ = strip_prefix(nama_kota)
    aliased = normalize(ALIAS.get(n, ALIAS.get(np_, '')))
    if aliased and (aliased in geo_keys_set or aliased in geo_keys_np_set):
        return aliased
    if n in geo_keys_set:
        return n
    if np_ in geo_keys_np_set:
        return np_
    for gk in geo_keys_np_set:
        if np_ == gk or (len(np_) >= 4 and (np_ in gk or gk in np_)):
            return gk
    return None


# ─────────────────────────────────────────────────────────────────────────────
# AGREGASI COUNTER → GDF  (dengan blacklist)
# ─────────────────────────────────────────────────────────────────────────────
def assign_freq_to_gdf(gdf_in, counter_kota, col_name):
    freq_map      = {}
    tidak_ketemu  = []
    diblacklist   = []

    for kota, freq in counter_kota.items():
        if is_blacklisted(kota):
            diblacklist.append((kota, freq))
            continue
        matched = cari_geo_key(kota)
        if matched:
            freq_map[matched] = freq_map.get(matched, 0) + freq
        else:
            tidak_ketemu.append((kota, freq))

    if diblacklist:
        print(f'\n🚫 [{col_name}] {len(diblacklist)} entri diblacklist:')
        for k, f in sorted(diblacklist, key=lambda x: -x[1])[:15]:
            print(f'   ✗ "{k}" (freq={f})')

    if tidak_ketemu:
        print(f'\n⚠️  [{col_name}] {len(tidak_ketemu)} kota tidak terpetakan:')
        for k, f in sorted(tidak_ketemu, key=lambda x: -x[1])[:20]:
            print(f'   - "{k}" (freq={f})')

    gdf_in[col_name] = gdf_in['geo_key_np'].map(freq_map).fillna(0).astype(int)
    return gdf_in

gdf = assign_freq_to_gdf(gdf, dict(counter_all),          'freq_all')
gdf = assign_freq_to_gdf(gdf, dict(counter_disrupsi),     'freq_disrupsi')
gdf = assign_freq_to_gdf(gdf, dict(counter_non_disrupsi), 'freq_non_disrupsi')

print('\n✅ Top-15 kota/kabupaten (semua artikel):')
print(
    gdf[gdf['freq_all'] > 0][[NAME_COL, 'freq_all', 'freq_disrupsi', 'freq_non_disrupsi']]
    .sort_values('freq_all', ascending=False).head(15).to_string(index=False)
)


# ─────────────────────────────────────────────────────────────────────────────
# LABEL LAUT, SELAT, SAMUDRA, DAN PERAIRAN INDONESIA
# Format: (longitude, latitude, teks, fontsize, rotasi, alpha)
# ─────────────────────────────────────────────────────────────────────────────
SEA_LABELS = [
    (97.0,  -14.5, 'SAMUDRA\nHINDIA',               13.0,   0, 0.85),
    (135.0,   5.0, 'SAMUDRA\nPASIFIK',              13.0,   0, 0.85),
    (107.5,  -5.5, 'Laut Jawa',                      11.0,   0, 0.80),
    (121.5,  -8.5, 'Laut Flores',                    10.0,   0, 0.78),
    (127.0,  -7.0, 'Laut Banda',                     10.0,   0, 0.78),
    (122.5,   1.5, 'Laut Maluku',                    10.0,   0, 0.78),
    (120.5,   4.5, 'Laut Sulawesi',                  10.0,   0, 0.78),
    (107.0,   5.5, 'Laut China\nSelatan',             10.0,   0, 0.78),
    (132.0,  -9.5, 'Laut Arafura',                   10.0,   0, 0.78),
    (134.5,  -2.5, 'Teluk\nCendrawasih',               9.5,   0, 0.75),
    (108.5,   3.0, 'Laut\nNatuna\nUtara',              9.5,   0, 0.75),
    (131.0,   1.0, 'Laut\nHalmahera',                  9.5,   0, 0.75),
    (124.5, -10.5, 'Laut Timor',                     10.0,   0, 0.78),
    (118.8,  -9.2, 'Laut\nSavu',                       9.5,   0, 0.75),
    (100.5,   2.5, 'Selat\nMalaka',                    9.5, -45, 0.75),
    (105.7,  -6.0, 'Selat\nSunda',                     9.0, -60, 0.75),
    (115.7,  -8.8, 'Selat\nLombok',                    8.0, -80, 0.72),
    (117.5,  -8.6, 'Selat\nAlas',                      8.0, -80, 0.72),
    (119.4,  -8.7, 'Selat\nSape',                      8.0, -80, 0.72),
    (116.6,   0.4, 'Selat\nMakassar',                  9.5, -80, 0.75),
    (125.0,   1.0, 'Selat\nManipa',                    8.0, -60, 0.70),
    (128.2,  -3.8, 'Selat\nSeram',                     8.0,   0, 0.70),
    (103.5,  -0.8, 'Selat\nBangka',                    8.0, -70, 0.70),
    (104.6,  -2.5, 'Selat\nGaspar',                    8.0, -60, 0.70),
    (105.8,   0.7, 'Selat\nKarimata',                  9.0, -70, 0.72),
    (120.5,  -1.0, 'Teluk\nTomini',                    9.0,   0, 0.72),
    (121.8,  -4.0, 'Teluk\nBone',                      9.0,   0, 0.72),
    (122.8,  -5.5, 'Teluk\nTolo',                      8.0,   0, 0.70),
    (133.0,  -3.5, 'Teluk\nBintuni',                   8.0,   0, 0.70),
    (119.0,  -4.0, 'Teluk\nMakassar',                  8.0,   0, 0.70),
]


# ─────────────────────────────────────────────────────────────────────────────
# COLORMAP & WARNA GLOBAL
# ─────────────────────────────────────────────────────────────────────────────
CMAP_CUSTOM = LinearSegmentedColormap.from_list(
    'GreenYellowRed',
    ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c'],
    N=512,
)

PAPER_BG   = '#A8D5E8'
SEA_CLR    = '#A8D5E8'
NO_DATA    = '#D6D6C2'
BORDER_CLR = '#4A4A4A'


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: filter SEA_LABELS yang masuk dalam extent peta
# ─────────────────────────────────────────────────────────────────────────────
def filter_sea_labels(extent: tuple, margin: float = 1.5) -> list:
    x0, y0, x1, y1 = extent
    return [
        lbl for lbl in SEA_LABELS
        if (x0 - margin) <= lbl[0] <= (x1 + margin)
        and (y0 - margin) <= lbl[1] <= (y1 + margin)
    ]


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: buat label nama kota yang clean (hapus prefix kab/kota)
# ─────────────────────────────────────────────────────────────────────────────
def clean_city_label(raw_name: str) -> str:
    """
    Buang prefix Kabupaten/Kota, kembalikan nama bersih Title Case.
    Contoh: 'Kabupaten Bandung' → 'Bandung'
             'Kota Surabaya'    → 'Surabaya'
    """
    n = raw_name.strip()
    for pfx in ['Kabupaten Administrasi ', 'Kota Administrasi ',
                'Kabupaten ', 'Kota ', 'Kab. ', 'Kab ']:
        if n.lower().startswith(pfx.lower()):
            n = n[len(pfx):]
            break
    return n.title()


# ─────────────────────────────────────────────────────────────────────────────
# FUNGSI PLOT CHOROPLETH — NASIONAL (paper-ready)
# ─────────────────────────────────────────────────────────────────────────────
def plot_choropleth_kota(
    gdf_map    : gpd.GeoDataFrame,
    col        : str,
    title      : str,
    filename   : str,
    name_col   : str   = NAME_COL,
    figsize    : tuple = (44, 18),
    top_label  : int   = 35,
    dpi        : int   = 300,
):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(PAPER_BG)
    ax.set_facecolor(SEA_CLR)

    vals        = gdf_map[col]
    vmax        = vals.max()
    mask_data   = vals > 0
    mask_nodata = ~mask_data

    gdf_map[mask_nodata].plot(
        ax=ax, color=NO_DATA,
        edgecolor=BORDER_CLR, linewidth=0.5,
    )

    if vmax > 0:
        vmax_data = int(gdf_map[mask_data][col].max())
        vmin_data = int(gdf_map[mask_data][col].min())
        norm = mcolors.Normalize(vmin=vmin_data, vmax=vmax_data)

        gdf_map[mask_data].plot(
            column=col, ax=ax,
            cmap=CMAP_CUSTOM, norm=norm,
            edgecolor=BORDER_CLR, linewidth=0.5,
            legend=False,
        )

        sm = plt.cm.ScalarMappable(cmap=CMAP_CUSTOM, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax,
                            fraction=0.014, pad=0.008, aspect=38, shrink=0.60)
        cbar.set_label('Frekuensi Kemunculan sebagai\nLokasi Kejadian',
                       fontsize=13, labelpad=10, color='#2c2c2c')
        cbar.ax.tick_params(labelsize=11, color='#2c2c2c', labelcolor='#2c2c2c')
        cbar.outline.set_edgecolor('#4A4A4A')
        for pos, lbl, clr in [(0.04, 'Rendah', '#1a9641'),
                               (0.50, 'Sedang', '#e65100'),
                               (0.94, 'Tinggi', '#b71c1c')]:
            cbar.ax.text(2.3, pos, lbl, transform=cbar.ax.transAxes,
                         fontsize=11, color=clr, va='center', fontweight='bold')

        # Anotasi angka saja (peta nasional terlalu kecil untuk nama kota)
        df_lab = (gdf_map[mask_data].copy()
                  .sort_values(col, ascending=False)
                  .head(top_label))
        for _, row in df_lab.iterrows():
            try:
                cx = row.geometry.centroid.x
                cy = row.geometry.centroid.y
                val = int(row[col])
                ratio   = val / vmax_data
                txt_clr = 'white' if ratio > 0.45 else '#111111'
                ax.annotate(
                    str(val), xy=(cx, cy),
                    fontsize=7.5, ha='center', va='center',
                    color=txt_clr, fontweight='bold', zorder=6,
                    path_effects=[pe.withStroke(
                        linewidth=2.0,
                        foreground='black' if txt_clr == 'white' else 'white')],
                )
            except Exception:
                pass

    for lon, lat, teks, fs, rot, alpha in SEA_LABELS:
        ax.text(lon, lat, teks,
                fontsize=fs, ha='center', va='center',
                color='#1a4a7a', fontstyle='italic',
                rotation=rot, alpha=alpha, zorder=4,
                path_effects=[pe.withStroke(linewidth=3.0, foreground='#A8D5E8')])

    patch_nodata = mpatches.Patch(facecolor=NO_DATA, edgecolor=BORDER_CLR,
                                  label='Tidak ada data')
    leg = ax.legend(handles=[patch_nodata], loc='lower left', fontsize=12,
                    framealpha=0.92, edgecolor='#4A4A4A', facecolor='#A8D5E8')
    for txt in leg.get_texts():
        txt.set_color('#1a1a2e')

    ax.set_title(title, fontsize=18, fontweight='bold',
                 pad=18, color='#1a1a2e', fontfamily='serif')

    minx, miny, maxx, maxy = gdf_map.total_bounds
    mx = (maxx - minx) * 0.005
    my = (maxy - miny) * 0.015
    ax.set_xlim(minx - mx, maxx + mx)
    ax.set_ylim(miny - my, maxy + my)

    ax.grid(True, linestyle='--', linewidth=0.2,
            color='#6A9AB0', alpha=0.30, zorder=1)
    ax.set_axis_off()
    plt.tight_layout(pad=1.0)
    plt.savefig(filename, bbox_inches='tight', dpi=dpi,
                facecolor=PAPER_BG, edgecolor='none')
    plt.show()
    print(f'✅ Tersimpan: {filename}')


# ─────────────────────────────────────────────────────────────────────────────
# RENDER 3 PETA NASIONAL
# ─────────────────────────────────────────────────────────────────────────────
MAPS = [
    dict(col='freq_all',
         title='Peta Sebaran Lokasi Kejadian Rantai Pasok per Kabupaten/Kota — Seluruh Artikel',
         filename='peta_kota_semua.png'),
    dict(col='freq_disrupsi',
         title='Peta Sebaran Lokasi Kejadian Rantai Pasok per Kabupaten/Kota — Artikel Disrupsi',
         filename='peta_kota_disrupsi.png'),
    dict(col='freq_non_disrupsi',
         title='Peta Sebaran Lokasi Kejadian Rantai Pasok per Kabupaten/Kota — Artikel Non-Disrupsi',
         filename='peta_kota_non_disrupsi.png'),
]

for m in MAPS:
    plot_choropleth_kota(gdf_map=gdf, **m)


# ═════════════════════════════════════════════════════════════════════════════
# PETA PER PULAU — Skala lokal, label nama kota
# ═════════════════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────────────────────
# MAPPING PROVINSI → PULAU  (inklusif varian ejaan GADM)
# ─────────────────────────────────────────────────────────────────────────────
PULAU_PROVINSI: dict = {
    'Sumatera': [
        'Aceh', 'Nanggroe Aceh Darussalam',
        'Sumatera Utara', 'Sumatera Barat', 'Riau',
        'Kepulauan Riau', 'Jambi', 'Bengkulu',
        'Sumatera Selatan', 'Sumatra Selatan',
        'Kepulauan Bangka Belitung', 'Kepulauan Bangka-Belitung',
        'Bangka-Belitung', 'Bangka Belitung',
        'Lampung',
    ],
    'Jawa': [
        'DKI Jakarta', 'Jakarta Raya',
        'Jawa Barat', 'Banten',
        'Jawa Tengah', 'DI Yogyakarta', 'Yogyakarta', 'Di Yogyakarta',
        'Jawa Timur',
    ],
    'Kalimantan': [
        'Kalimantan Barat', 'Kalimantan Tengah',
        'Kalimantan Selatan', 'Kalimantan Timur', 'Kalimantan Utara',
    ],
    'Sulawesi': [
        'Sulawesi Utara', 'Gorontalo', 'Sulawesi Tengah',
        'Sulawesi Barat', 'Sulawesi Selatan', 'Sulawesi Tenggara',
    ],
    'Bali & Nusa Tenggara': [
        'Bali',
        'Nusa Tenggara Barat', 'NTB', 'West Nusa Tenggara',
        'Nusa Tenggara Timur', 'NTT', 'East Nusa Tenggara',
    ],
    'Maluku': [
        'Maluku', 'Maluku Utara',
    ],
    'Papua': [
        'Papua', 'Papua Barat', 'Papua Tengah',
        'Papua Pegunungan', 'Papua Selatan', 'Papua Barat Daya',
        'Irian Jaya Barat',
    ],
}

# ─────────────────────────────────────────────────────────────────────────────
# FALLBACK BOUNDING-BOX — diperluas agar tidak ada yang terpotong
# ─────────────────────────────────────────────────────────────────────────────
PULAU_BBOX: dict = {
    #                    minLon  minLat  maxLon  maxLat
    'Sumatera'            : ( 94.5,  -6.5, 109.5,  6.5),
    'Jawa'                : (105.0,  -9.2, 115.5, -4.8),
    'Kalimantan'          : (107.5,  -4.8, 119.0,  7.8),
    # ↓ miny diperluas ke -8.5 agar Sulsel / Sultra / Selayar masuk
    'Sulawesi'            : (118.5,  -8.5, 125.5,  3.0),
    # ↓ maxy dinaikkan ke -7.5, miny ke -11.5 agar semua kab Bali–NTT masuk
    'Bali & Nusa Tenggara': (114.0, -11.5, 125.5, -7.5),
    'Maluku'              : (124.0,  -9.0, 135.5,  3.5),
    'Papua'               : (130.0,  -9.5, 141.5,  1.0),
}

# ─────────────────────────────────────────────────────────────────────────────
# KONFIGURASI FIGSIZE per pulau
# ─────────────────────────────────────────────────────────────────────────────
PULAU_CONFIG: dict = {
    'Sumatera'            : dict(figsize=(18, 28), top_label=20, margin_x=0.03, margin_y=0.02),
    'Jawa'                : dict(figsize=(30, 14), top_label=30, margin_x=0.02, margin_y=0.06),
    'Kalimantan'          : dict(figsize=(20, 22), top_label=20, margin_x=0.03, margin_y=0.02),
    # ↓ lebih tinggi agar bentuk K tidak terpotong
    'Sulawesi'            : dict(figsize=(20, 28), top_label=20, margin_x=0.04, margin_y=0.03),
    # ↓ lebih lebar, rasio ~3:1
    'Bali & Nusa Tenggara': dict(figsize=(32, 12), top_label=15, margin_x=0.02, margin_y=0.06),
    'Maluku'              : dict(figsize=(20, 22), top_label=15, margin_x=0.04, margin_y=0.03),
    'Papua'               : dict(figsize=(26, 18), top_label=20, margin_x=0.02, margin_y=0.03),
}


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: ambil sub-GDF untuk satu pulau
# ─────────────────────────────────────────────────────────────────────────────
def get_gdf_pulau(gdf_full: gpd.GeoDataFrame, nama_pulau: str) -> gpd.GeoDataFrame:
    """
    Filter GDF ke kabupaten/kota pada satu pulau/region.
    Prioritas 1 → kolom NAME_1 (provinsi) dari GADM.
    Prioritas 2 → bounding-box centroid (fallback).
    """
    prov_list = PULAU_PROVINSI.get(nama_pulau, [])

    if 'NAME_1' in gdf_full.columns and prov_list:
        prov_norm  = {normalize(p) for p in prov_list}
        name1_norm = gdf_full['NAME_1'].apply(lambda x: normalize(str(x)))
        mask = name1_norm.isin(prov_norm)
        sub  = gdf_full[mask].copy()
        found = set(name1_norm[mask].unique())
        print(f'   NAME_1 cocok ({len(sub)} baris): {sorted(found)}')
        if len(sub) > 0:
            return sub
        # Debug: tampilkan semua NAME_1 unik untuk diagnosis
        print(f'   ⚠️  0 cocok. Semua NAME_1: '
              f'{sorted(gdf_full["NAME_1"].apply(lambda x: normalize(str(x))).unique())}')

    # Fallback bbox
    bbox = PULAU_BBOX.get(nama_pulau)
    if bbox is None:
        raise ValueError(f'Pulau tidak dikenal: {nama_pulau}')
    minx, miny, maxx, maxy = bbox
    cx   = gdf_full.geometry.centroid.x
    cy   = gdf_full.geometry.centroid.y
    mask = (cx >= minx) & (cx <= maxx) & (cy >= miny) & (cy <= maxy)
    sub  = gdf_full[mask].copy()
    print(f'   BBOX fallback: {len(sub)} baris (lon {minx}–{maxx}, lat {miny}–{maxy})')
    return sub


# ─────────────────────────────────────────────────────────────────────────────
# FUNGSI PLOT CHOROPLETH PER PULAU — skala lokal + label nama kota
# ─────────────────────────────────────────────────────────────────────────────
def plot_choropleth_pulau(
    gdf_pulau  : gpd.GeoDataFrame,
    col        : str,
    title      : str,
    filename   : str,
    name_col   : str   = NAME_COL,
    figsize    : tuple = (22, 20),
    top_label  : int   = 20,
    dpi        : int   = 300,
    margin_x   : float = 0.03,
    margin_y   : float = 0.03,
):
    """
    Plot choropleth satu pulau.
    • Skala colorbar = nilai tertinggi di pulau tersebut (LOKAL).
    • Label NAMA KOTA ditampilkan untuk semua kab/kota yang punya data (freq > 0).
      Format: "Nama Kota\n(angka)" di tengah poligon.
    • Untuk kab/kota tanpa data, hanya nama kota yang ditampilkan (tanpa angka),
      dengan fontsize lebih kecil dan warna abu-abu.
    """
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(PAPER_BG)
    ax.set_facecolor(SEA_CLR)

    vals        = gdf_pulau[col]
    mask_data   = vals > 0
    mask_nodata = ~mask_data

    # ── Wilayah tanpa data ───────────────────────────────────────────────────
    gdf_pulau[mask_nodata].plot(
        ax=ax, color=NO_DATA,
        edgecolor=BORDER_CLR, linewidth=0.6,
    )

    vmax = vals.max()

    if vmax > 0:
        vmax_data = int(gdf_pulau[mask_data][col].max())
        vmin_data = int(gdf_pulau[mask_data][col].min())
        norm = mcolors.Normalize(vmin=vmin_data, vmax=vmax_data)

        gdf_pulau[mask_data].plot(
            column=col, ax=ax,
            cmap=CMAP_CUSTOM, norm=norm,
            edgecolor=BORDER_CLR, linewidth=0.6,
            legend=False,
        )

        # ── Colorbar ─────────────────────────────────────────────────────────
        sm = plt.cm.ScalarMappable(cmap=CMAP_CUSTOM, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax,
                            fraction=0.016, pad=0.01, aspect=35, shrink=0.55)
        cbar.set_label('Frekuensi Kemunculan sebagai\nLokasi Kejadian',
                       fontsize=12, labelpad=10, color='#2c2c2c')
        cbar.ax.tick_params(labelsize=10, color='#2c2c2c', labelcolor='#2c2c2c')
        cbar.outline.set_edgecolor('#4A4A4A')
        for pos, lbl, clr in [(0.04, 'Rendah', '#1a9641'),
                               (0.50, 'Sedang', '#e65100'),
                               (0.94, 'Tinggi', '#b71c1c')]:
            cbar.ax.text(2.3, pos, lbl, transform=cbar.ax.transAxes,
                         fontsize=10, color=clr, va='center', fontweight='bold')
        cbar.ax.set_title(f'Maks: {vmax_data}',
                          fontsize=10, color='#b71c1c', fontweight='bold', pad=6)

    # ── Extent peta ──────────────────────────────────────────────────────────
    minx, miny, maxx, maxy = gdf_pulau.total_bounds
    mx  = (maxx - minx) * margin_x
    my  = (maxy - miny) * margin_y
    ext = (minx - mx, miny - my, maxx + mx, maxy + my)
    ax.set_xlim(ext[0], ext[2])
    ax.set_ylim(ext[1], ext[3])

    # ── Estimasi ukuran pixel per derajat (untuk filter area minimum) ────────
    fig_w_in, fig_h_in = figsize
    lon_range = ext[2] - ext[0]
    lat_range = ext[3] - ext[1]
    px_per_deg_lon = (fig_w_in * dpi) / lon_range
    px_per_deg_lat = (fig_h_in * dpi) / lat_range
    MIN_AREA_DEG2  = (12 / px_per_deg_lon) * (12 / px_per_deg_lat)  # ~12×12 px

    # ═════════════════════════════════════════════════════════════════════════
    # LABEL NAMA KOTA
    # Strategi:
    #   A) Kab/kota BERDATA (freq > 0)  → tampilkan "NamaKota\n(N)"
    #      - Baris atas  : nama kota (bold)
    #      - Baris bawah : angka dalam kurung (bold, ukuran sama)
    #      - Warna teks  : putih (jika ratio > 0.45) else hitam gelap
    #      - Skip jika poligon terlalu kecil (area < MIN_AREA_DEG2)
    #   B) Kab/kota TANPA DATA        → tampilkan nama saja (abu, fontsize kecil)
    #      - Hanya untuk top_label terbesar berdasarkan area
    # ═════════════════════════════════════════════════════════════════════════

    # -- A: kab/kota berdata --------------------------------------------------
    if vmax > 0:
        df_berdata = gdf_pulau[mask_data].copy()
        for _, row in df_berdata.iterrows():
            try:
                geom = row.geometry
                if geom is None or geom.is_empty:
                    continue
                area = geom.area
                if area < MIN_AREA_DEG2:
                    continue
                cx_pt = geom.centroid.x
                cy_pt = geom.centroid.y
                val   = int(row[col])
                ratio = val / vmax_data
                txt_clr = 'white' if ratio > 0.45 else '#111111'
                stroke  = 'black' if txt_clr == 'white' else 'white'
                nama    = clean_city_label(str(row[name_col]))
                label   = f'{nama}\n({val})'
                ax.annotate(
                    label, xy=(cx_pt, cy_pt),
                    fontsize=8, ha='center', va='center',
                    color=txt_clr, fontweight='bold',
                    multialignment='center', zorder=7,
                    path_effects=[pe.withStroke(linewidth=2.5, foreground=stroke)],
                )
            except Exception:
                pass

    # -- B: kab/kota tanpa data — tampilkan nama saja (top area terbesar) ----
    df_nodata = gdf_pulau[mask_nodata].copy()
    df_nodata['_area'] = df_nodata.geometry.area
    df_nodata_show = df_nodata.nlargest(top_label, '_area')
    for _, row in df_nodata_show.iterrows():
        try:
            geom = row.geometry
            if geom is None or geom.is_empty:
                continue
            area = geom.area
            if area < MIN_AREA_DEG2 * 0.7:
                continue
            cx_pt = geom.centroid.x
            cy_pt = geom.centroid.y
            nama  = clean_city_label(str(row[name_col]))
            ax.annotate(
                nama, xy=(cx_pt, cy_pt),
                fontsize=6.5, ha='center', va='center',
                color='#555555', fontweight='normal',
                multialignment='center', zorder=5,
                path_effects=[pe.withStroke(linewidth=1.8, foreground='white')],
            )
        except Exception:
            pass

    # ── Label laut dalam extent ───────────────────────────────────────────────
    for lon, lat, teks, fs, rot, alpha in filter_sea_labels(ext):
        ax.text(lon, lat, teks,
                fontsize=fs, ha='center', va='center',
                color='#1a4a7a', fontstyle='italic',
                rotation=rot, alpha=alpha, zorder=4,
                path_effects=[pe.withStroke(linewidth=3.0, foreground='#A8D5E8')])

    # ── Legend ───────────────────────────────────────────────────────────────
    patch_nodata = mpatches.Patch(facecolor=NO_DATA, edgecolor=BORDER_CLR,
                                  label='Tidak ada data')
    leg = ax.legend(handles=[patch_nodata], loc='lower left', fontsize=11,
                    framealpha=0.92, edgecolor='#4A4A4A', facecolor='#A8D5E8')
    for txt in leg.get_texts():
        txt.set_color('#1a1a2e')

    ax.set_title(title, fontsize=15, fontweight='bold',
                 pad=14, color='#1a1a2e', fontfamily='serif')
    ax.grid(True, linestyle='--', linewidth=0.2,
            color='#6A9AB0', alpha=0.30, zorder=1)
    ax.set_axis_off()
    plt.tight_layout(pad=1.0)
    plt.savefig(filename, bbox_inches='tight', dpi=dpi,
                facecolor=PAPER_BG, edgecolor='none')
    plt.show()
    print(f'✅ Tersimpan: {filename}')


# ─────────────────────────────────────────────────────────────────────────────
# DEFINISI KOMBINASI PETA × PULAU
# ─────────────────────────────────────────────────────────────────────────────
MAPS_PULAU = [
    dict(col='freq_all',          suffix='semua',        label='Seluruh Artikel'),
    dict(col='freq_disrupsi',     suffix='disrupsi',     label='Artikel Disrupsi'),
    dict(col='freq_non_disrupsi', suffix='non_disrupsi', label='Artikel Non-Disrupsi'),
]

# ─────────────────────────────────────────────────────────────────────────────
# RENDER SEMUA PETA PER PULAU  (7 pulau × 3 jenis = 21 gambar)
# ─────────────────────────────────────────────────────────────────────────────
print('\n' + '═' * 60)
print('🗺️  RENDER PETA PER PULAU')
print('═' * 60)

for nama_pulau, cfg in PULAU_CONFIG.items():
    print(f'\n📍 {nama_pulau}')
    try:
        gdf_sub = get_gdf_pulau(gdf, nama_pulau)
        n_kab   = len(gdf_sub)
        print(f'   → {n_kab} kabupaten/kota')

        if n_kab == 0:
            print('   ⚠️  Tidak ada geometri, dilewati.')
            continue

        for m in MAPS_PULAU:
            slug  = (nama_pulau.lower()
                     .replace('&', 'dan')
                     .replace('/', '_')
                     .replace(' ', '_'))
            fname = f"peta_{slug}_{m['suffix']}.png"
            title = (
                f"Sebaran Lokasi Kejadian Rantai Pasok — {nama_pulau}\n"
                f"({m['label']})"
            )

            sub_vals  = gdf_sub[m['col']]
            n_berdata = (sub_vals > 0).sum()
            if n_berdata > 0:
                print(f"   [{m['suffix']}] {n_berdata} kab/kota berdata | "
                      f"Maks: {int(sub_vals.max())} | "
                      f"Total: {int(sub_vals.sum())}")
            else:
                print(f"   [{m['suffix']}] ⚠️  Semua nilai = 0, tetap diplot.")

            plot_choropleth_pulau(
                gdf_pulau = gdf_sub,
                col       = m['col'],
                title     = title,
                filename  = fname,
                **cfg,
            )

    except Exception as e:
        print(f'   ❌ Gagal: {e}')
        traceback.print_exc()

print('\n✅ Semua peta per pulau selesai!')
print(f'   Total file: {len(PULAU_CONFIG) * len(MAPS_PULAU)} gambar')


# ─────────────────────────────────────────────────────────────────────────────
# RINGKASAN ANALISIS
# ─────────────────────────────────────────────────────────────────────────────
print('\n✅ Semua peta berhasil dibuat!')
print('=' * 60)
print('📌 RINGKASAN ANALISIS SUPPLY CHAIN DISRUPSI')
print('=' * 60)

print('\n📊 Total Artikel per Label:')
print(df['Kategori_LLM'].value_counts().to_string())

print('\n🏙️ Top 10 Kota Paling Sering Disebut (Semua Artikel):')
print(df_kota_all.head(10).to_string(index=False))

print('\n🔴 Top 10 Kota dalam Artikel Disrupsi:')
print(df_kota_disrupsi.head(10).to_string(index=False))

print('\n📅 Tahun dengan Artikel Disrupsi Terbanyak:')
print(tahun_disrupsi.sort_values(ascending=False).to_string())

print('\n✅ Analisis selesai!')